In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

import cv2
from tqdm import tqdm_notebook, tnrange
from glob import glob
from itertools import chain
from skimage.io import imread, imshow, concatenate_images
from skimage.transform import resize
from skimage.morphology import label
from sklearn.model_selection import train_test_split

import tensorflow as tf
from skimage.color import rgb2gray
# from tensorflow.keras import Input
from tensorflow.keras.models import Model, load_model, save_model
from tensorflow.keras.layers import Input, Activation, BatchNormalization, Dropout, Lambda, Conv2D, Conv2DTranspose, MaxPooling2D, concatenate, add
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.preprocessing import StandardScaler, normalize
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.python.keras import Sequential
from tensorflow.keras import layers, optimizers
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, LearningRateScheduler
import tensorflow.keras.backend as K
from IPython.display import display
import skimage.io
from skimage import io
import keras
import tensorflow as tf
import tensorflow.keras.backend as K
from IPython.display import display

In [ ]:

im_width = 256
im_height = 256



train_files = []
mask_files = glob('../input//masks*/*_segmentation*')


for i in mask_files:
    train_files.append(i.replace('_segmentation','').replace('/masks', '/images').replace('.png','.jpg'))

In [ ]:
df = pd.DataFrame({"image_path": train_files, "mask_path":mask_files})

def diagnosis(mask_path):
    value = np.max(cv2.imread(mask_path))
    if value:
        return 1
    else:
        return 0

df['mask'] = df["mask_path"].apply(lambda x: diagnosis(x))
df.head()

In [ ]:

image1 = io.imread(df.image_path[1])

image2 = io.imread(df.image_path[2])

# Show masks
mask1 = io.imread(df.mask_path[1])
     
mask2 = io.imread(df.mask_path[2])


f, axarr = plt.subplots(2,2,figsize=(20, 20))
axarr[0,0].imshow(image1)
axarr[0,0].title.set_text("skin image")
axarr[0,1].imshow(mask1,cmap='gray')
axarr[0,1].title.set_text("skin mask")
matplotlib.rcParams.update({'font.size': 22})
axarr[1,0].imshow(image2)
axarr[1,1].imshow(mask2,cmap='gray')

In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.15, random_state=42)
df_train, df_val = train_test_split(df_train, test_size=0.15, random_state=42)
print(df_train.values.shape)
print(df_val.values.shape)
print(df_test.values.shape)

In [ ]:
def train_generator(data_frame, batch_size, aug_dict,
        image_color_mode="rgb",
        mask_color_mode="grayscale",
        image_save_prefix="image",
        mask_save_prefix="mask",
        save_to_dir=None,
        target_size=(224,224),
        seed=1):

    image_datagen = ImageDataGenerator(**aug_dict)
    mask_datagen = ImageDataGenerator(**aug_dict)
    
    image_generator = image_datagen.flow_from_dataframe(
        data_frame,
        x_col = "image_path",
        class_mode = None,
        color_mode = image_color_mode,
        target_size = target_size,
        batch_size = batch_size,
        save_to_dir = save_to_dir,
        save_prefix  = image_save_prefix,
        seed = seed)

    mask_generator = mask_datagen.flow_from_dataframe(
        data_frame,
        x_col = "mask_path",
        class_mode = None,
        color_mode = mask_color_mode,
        target_size = target_size,
        batch_size = batch_size,
        save_to_dir = save_to_dir,
        save_prefix  = mask_save_prefix,
        seed = seed)

    train_gen = zip(image_generator, mask_generator)
    
    for (img, mask) in train_gen:
        img, mask = adjust_data(img, mask)
        yield (img,mask)

'''def adjust_data(img,mask):
    img = img / 255.
    mask = mask / 255.
    mask[mask > 0.5] = 1
    mask[mask <= 0.5] = 0
    
    return (img, mask)'''

In [ ]:
def adjust_data(img, mask):
    """Normalize images using the same approach as PyTorch's transforms.Normalize."""
    img = img / 255.0  # Scale to [0,1]

    # Normalize using mean and std (PyTorch equivalent)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std  # Normalize per channel

    mask = mask / 255.0  # Scale to [0,1]
    mask[mask > 0.5] = 1
    mask[mask <= 0.5] = 0
    
    return img, mask

In [ ]:
smooth=100

def dice_coef(y_true, y_pred):
    y_true = K.flatten(y_true)
    y_pred = K.flatten(y_pred)
    intersection = K.sum(y_true * y_pred)
    union = K.sum(y_true) + K.sum(y_pred)
    return (2.0 * intersection + smooth) / (union + smooth)

def dice_coef_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
    return dice_coef_loss(y_true, y_pred) + bce(y_true, y_pred)

def iou(y_true, y_pred):
    intersection = K.sum(y_true * y_pred)
    sum_ = K.sum(y_true + y_pred)
    jac = (intersection + smooth) / (sum_ - intersection + smooth)
    return jac

In [ ]:
#Importing necessary dependancies
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
from glob import glob
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Input
from tensorflow.keras.models import Model, load_model, save_model ,Sequential
from tensorflow.keras.layers import Input, Activation, BatchNormalization, Dropout, Lambda, Conv2D,MaxPooling2D, concatenate, Dense, Flatten, UpSampling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
import tensorflow as tf
import tensorflow.keras.layers as L
from tensorflow.keras.models import Model
from tensorflow.keras.applications import resnet50
#model.trainable = False  

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer

class WeightedCombination(Layer):
    def __init__(self, **kwargs):
        super(WeightedCombination, self).__init__(**kwargs)

    def build(self, input_shape):
        # Initialize learnable weights for each tensor
        self.w1 = self.add_weight(name='w1', shape=(1,), initializer='ones', trainable=True)
        self.w2 = self.add_weight(name='w2', shape=(1,), initializer='ones', trainable=True)
        self.w3 = self.add_weight(name='w3', shape=(1,), initializer='ones', trainable=True)
        self.w4 = self.add_weight(name='w4', shape=(1,), initializer='ones', trainable=True)

    def call(self, inputs):
        # Apply learnable weights
        nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4 = inputs
        weighted_output = (self.w4 * (nestnet_output_4 - nestnet_output_1 - nestnet_output_2 - nestnet_output_3))
        return weighted_output


In [ ]:
### new NEW NEW fusion block

def fusion_tasks(conv1_2, conv1_3, conv1_4, conv1_5):
    
    from tensorflow.keras.layers import UpSampling2D, Conv2D

    
    nestnet_output_1 = UpSampling2D(size=(2, 2))(conv1_2)  # Upsample by a factor of 2
    nestnet_output_1 = UpSampling2D(size=(2, 2))(nestnet_output_1)  # Upsample by a factor of 2
    nestnet_output_1 = UpSampling2D(size=(2, 2))(nestnet_output_1)  # Upsample by a factor of 2
    nestnet_output_1 = Conv2D(64, (1, 1), padding='same', activation='sigmoid')(nestnet_output_1)  # Apply Conv2D


    nestnet_output_2 = UpSampling2D(size=(2, 2))(conv1_3)  # Upsample by a factor of 2
    nestnet_output_2 = UpSampling2D(size=(2, 2))(nestnet_output_2)  # Upsample by a factor of 2
    nestnet_output_2 = Conv2D(64, (1, 1), padding='same', activation='sigmoid')(nestnet_output_2)  # Apply Conv2D


    nestnet_output_3 = UpSampling2D(size=(2, 2))(conv1_4)  # Upsample by a factor of 2 
    nestnet_output_3 = Conv2D(64, (1, 1), padding='same', activation='sigmoid')(nestnet_output_3)  # Apply Conv2D


    nestnet_output_4 = conv1_5
    
    
    nestnet_output_1_1 = tf.keras.layers.Concatenate(axis=-1)([nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4])
    nestnet_output_1_1 = Conv2D(64, (1, 1), name='output_1_1',padding='same')(nestnet_output_1_1)
    nestnet_output_1_2 = WeightedCombination()([nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4])

    nestnet_output_1 = tf.keras.layers.Add()([nestnet_output_1_1, nestnet_output_1_2])
    nestnet_output_1 = Conv2D(64, (1, 1), padding='same')(nestnet_output_1)
    
    nestnet_output_2_1 = tf.keras.layers.Concatenate(axis=-1)([nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4])
    nestnet_output_2_1 = Conv2D(64, (1, 1), name='output_2_1',padding='same')(nestnet_output_2_1)
    nestnet_output_2_2 = WeightedCombination()([nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4])

    nestnet_output_2 = tf.keras.layers.Add()([nestnet_output_2_1, nestnet_output_2_2])
    nestnet_output_2 = Conv2D(64, (1, 1), padding='same')(nestnet_output_2)
    
    nestnet_output_3_1 = tf.keras.layers.Concatenate(axis=-1)([nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4])
    nestnet_output_3_1 = Conv2D(64, (1, 1), name='output_3_1',padding='same')(nestnet_output_3_1)
    nestnet_output_3_2 = WeightedCombination()([nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4])

    nestnet_output_3 = tf.keras.layers.Add()([nestnet_output_3_1, nestnet_output_3_2])
    nestnet_output_3 = Conv2D(64, (1, 1), padding='same')(nestnet_output_3)

    nestnet_output_4_1 = tf.keras.layers.Concatenate(axis=-1)([nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4])
    nestnet_output_4_1 = Conv2D(64, (1, 1), name='output_4_1',padding='same')(nestnet_output_4_1)
    nestnet_output_4_2 = WeightedCombination()([nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4])

    nestnet_output_4 = tf.keras.layers.Add()([nestnet_output_4_1, nestnet_output_4_2])
    nestnet_output_4 = Conv2D(64, (1, 1), padding='same')(nestnet_output_4)
        
    return nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4

In [ ]:
#### Multi-branch fusion attention (MFA) module #####

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.signal import fft2d, dct

class GlobalMinPooling2D(layers.Layer):
    def __init__(self, **kwargs):
        super(GlobalMinPooling2D, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2])

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        config = super(GlobalMinPooling2D, self).get_config()
        return config

class DeeperGlobalLocalAttentionLayer1(layers.Layer):
    def __init__(self, units, activation='sigmoid', dropout_rate=0.2, use_scale=True, axis=-1, **kwargs):
        super(DeeperGlobalLocalAttentionLayer1, self).__init__(**kwargs)
        self.units = units
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_scale = use_scale
        self.axis = axis

    def build(self, input_shape):
        _, _, _, channels = input_shape
        
        
        self.global_conv1 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.global_avg_pooling1 = layers.GlobalAveragePooling2D()
        
        self.global_conv2 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.global_avg_pooling2 = layers.GlobalMaxPooling2D()
        
        self.global_conv_3 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.global_avg_pooling_3 = GlobalMinPooling2D()
        
        
        self.global_conv3 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.global_avg_pooling3 = layers.GlobalAveragePooling2D()
        
        self.global_conv4 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.global_avg_pooling4 = layers.GlobalMaxPooling2D()

        self.global_conv_4 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.global_avg_pooling_4 = GlobalMinPooling2D()
        
        
        self.concat1 = layers.Add()
        self.concat2 = layers.Add()
        self.concat3 = layers.Add()
        self.concat4 = layers.Add()
        self.concat_3 = layers.Add()
        self.concat_4 = layers.Add()
        
        self.concat5 = layers.Concatenate(axis=-1)
        
        self.global_attention = layers.Dense(units=self.units, activation=self.activation)

        #self.local_dsc = layers.DepthwiseConv2D(kernel_size=(3, 3), padding="same", depth_multiplier=1)
        self.local_conv1 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.local_conv2 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.concat6 = layers.Add()
        
        if self.use_scale:
            self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale')
            self.local_scale = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale')
        
        super(DeeperGlobalLocalAttentionLayer1, self).build(input_shape)

    def call(self, inputs, training=None):
        ##### Hierarchical Information Fusion Attention(HIFA) ######
        
        global_attention1 = self.global_conv1(inputs)
        global_avg1 = self.global_avg_pooling1(global_attention1)
        
        global_attention2 = self.global_conv2(global_attention1)
        global_avg2 = self.global_avg_pooling2(global_attention2)

        global_attention_3 = self.global_conv_3(global_attention1)
        global_avg_3 = self.global_avg_pooling_3(global_attention_3)
        
        
        global_concat1 = self.concat1([global_avg1, global_avg2, global_avg_3])
        global_sub1 = global_avg2 - global_avg1 - global_avg_3
        global_concat1 = global_concat1 + global_sub1
        
        global_attention_concat1 = self.concat2([global_attention1, global_attention2, global_attention_3])
        
        global_sub_1 = global_attention2 - global_attention1 - global_attention_3

        global_attention_concat1 = global_attention_concat1 + global_sub_1
        
        global_attention3 = self.global_conv3(global_attention_concat1)
        global_avg3 = self.global_avg_pooling3(global_attention3)
        
        global_attention4 = self.global_conv4(global_attention3)
        global_avg4 = self.global_avg_pooling4(global_attention4)

        global_attention_4 = self.global_conv_3(global_attention3)
        global_avg_4 = self.global_avg_pooling_3(global_attention_4)
        
        
        global_concat2 = self.concat3([global_avg3, global_avg4, global_avg_4])
        global_sub2 = global_avg4 - global_avg3 - global_avg_4
        global_concat2 = global_concat2 + global_sub2

        
        #global_attention_concat2 = self.concat4([global_attention3, global_attention4, global_attention_4])
        #global_sub_2 = global_attention4 - global_attention3 - global_attention_4

        #global_attention_concat2 = global_attention_concat2 + global_sub_2

        
        
        global_avg_concat = self.concat5([global_concat1, global_concat2])
        
        global_attention = self.global_attention(global_avg_concat)
        global_attention = tf.expand_dims(tf.expand_dims(global_attention, 1), 1)

        ##### frequency domain Local Information learning Attention ######
        
        input_shape = tf.shape(inputs)

        
        flattened_inputs = tf.reshape(inputs, [-1, input_shape[-1]])  # Flatten along the last axis
        dct_transformed = tf.signal.dct(flattened_inputs, type=2, norm='ortho')
        dct_transformed = tf.reshape(dct_transformed, input_shape)  # Reshape back to original dimensions
        
        local_attention1 = self.local_conv1(dct_transformed)
        local_attention1 = tf.reduce_mean(local_attention1, axis=[1, 2], keepdims=True)  # Reduce spatial dimensions
        local_attention2 = self.local_conv2(local_attention1)
        local_attention2 = tf.reduce_mean(local_attention2, axis=[1, 2], keepdims=True)  # Reduce spatial dimensions

        local_attention3 = self.local_conv1(inputs)
        local_attention3 = tf.reduce_max(local_attention3, axis=[1, 2], keepdims=True)  # Reduce spatial dimensions
        local_attention4 = self.local_conv2(local_attention3)
        local_attention4 = tf.reduce_max(local_attention4, axis=[1, 2], keepdims=True)  # Reduce spatial dimensions

        local_attention5 = self.local_conv1(inputs)
        local_attention5 = tf.reduce_mean(local_attention5, axis=[1, 2], keepdims=True)  # Reduce spatial dimensions
        local_attention6 = self.local_conv2(local_attention5)
        local_attention6 = tf.reduce_mean(local_attention6, axis=[1, 2], keepdims=True)  # Reduce spatial dimensions

        local_attention7 = self.local_conv1(dct_transformed)
        local_attention7 = tf.reduce_max(local_attention7, axis=[1, 2], keepdims=True)  # Reduce spatial dimensions
        local_attention8 = self.local_conv2(local_attention7)
        local_attention8 = tf.reduce_max(local_attention8, axis=[1, 2], keepdims=True)  # Reduce spatial dimensions

        
        
        #local_attention = self.concat6([local_attention1, local_attention2, local_attention3, local_attention4, 
         #                              local_attention5, local_attention6, local_attention7, local_attention8])
        
        local_attention_avg = (self.concat6([local_attention1, local_attention2, local_attention5, local_attention6])) 
        local_attention_max = self.concat6([local_attention3, local_attention4, local_attention7, local_attention8]) 

        local_attention = L.Activation("relu")(self.concat6([local_attention_avg, local_attention_max])) 
        
        # Scale Global and Local Attention
        if self.use_scale:
            global_attention *= self.global_scale
            local_attention *= self.local_scale

        # Combine Global and Local Attention
        attention = tf.sigmoid(global_attention + local_attention)
        return attention

    def get_config(self):
        config = super(DeeperGlobalLocalAttentionLayer1, self).get_config()
        config.update({'units': self.units, 'activation': self.activation, 'dropout_rate': self.dropout_rate,
                       'use_scale': self.use_scale})
        return config

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.signal import fft2d, dct

class GlobalMinPooling2D(layers.Layer):
    def __init__(self, **kwargs):
        super(GlobalMinPooling2D, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2])

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        config = super(GlobalMinPooling2D, self).get_config()
        return config


import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, Input


class FrequencyTransformLayer(layers.Layer):
    def __init__(self, transform_type='dct', **kwargs):
        super(FrequencyTransformLayer, self).__init__(**kwargs)
        self.transform_type = transform_type

    def call(self, inputs):
        # Ensure inputs are float32 for DCT and FFT operations
        inputs = tf.cast(inputs, tf.float32)

        if self.transform_type == 'dct':
            # Apply 2D DCT along the last axis
            input_shape = tf.shape(inputs)
            flattened_inputs = tf.reshape(inputs, [-1, input_shape[-1]])  # Flatten along the last axis
            dct_transformed = tf.signal.dct(flattened_inputs, type=2, norm='ortho')
            return tf.reshape(dct_transformed, input_shape)  # Reshape back to original dimensions
        elif self.transform_type == 'fft':
            # Apply 2D FFT and return the magnitude
            fft_transformed = tf.signal.fft2d(tf.cast(inputs, tf.complex64))
            return tf.math.abs(fft_transformed)
        else:
            raise ValueError("Unsupported transform type. Choose 'dct' or 'fft'.")

    def get_config(self):
        config = super(FrequencyTransformLayer, self).get_config()
        config.update({'transform_type': self.transform_type})
        return config


class DeeperGlobalLocalAttentionLayerWithFrequency(layers.Layer):
    def __init__(self, units, activation='sigmoid', dropout_rate=0.3, use_scale=True, axis=-1, **kwargs):
        super(DeeperGlobalLocalAttentionLayerWithFrequency, self).__init__(**kwargs)
        self.units = units
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_scale = use_scale
        self.axis = axis

        # Frequency Transform Layers
        self.dct_transform = FrequencyTransformLayer(transform_type='dct')
        self.fft_transform = FrequencyTransformLayer(transform_type='fft')

        # Define pooling and dense layers for frequency features
        self.global_avg_pooling = layers.GlobalAveragePooling2D()
        self.global_min_pooling = GlobalMinPooling2D()
        self.global_max_pooling = layers.GlobalMaxPooling2D()
        self.global_attention_freq = layers.Dense(units=self.units, activation=self.activation)
        self.local_attn = DeeperGlobalLocalAttentionLayer1(units=self.units, activation='sigmoid', 
                                                                              dropout_rate=0.2,  # You can adjust the dropout rate
                                                                              use_scale=self.use_scale)


    def build(self, input_shapes):
        input_shape1, input_shape2 = input_shapes
        _, _, _, channels1 = input_shape1
        _, _, _, channels2 = input_shape2

        # Global and Local Layers
        self.global_min_pooling1 = GlobalMinPooling2D()
        self.global_avg_pooling1 = layers.GlobalAveragePooling2D()
        self.global_max_pooling1 = layers.GlobalMaxPooling2D()
        self.batch = BatchNormalization()
        
        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)

        self.local_conv1 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)

        # Scale Weights
        if self.use_scale:
            #self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale')
            self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer=RandomNormal(mean=1.0, stddev=0.02), trainable=True, name='global_scale', 
                                                constraint=tf.keras.constraints.MaxNorm(2.0))
            self.global_scale2 = self.add_weight(shape=(1, 1, 1, 1),  initializer=HeNormal(), trainable=True, name='global_scale2', constraint=tf.keras.constraints.MaxNorm(2.0))
            
            self.local_scale = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale', constraint=tf.keras.constraints.MaxNorm(2.0))
            self.freq_scale = self.add_weight(shape=(1, 1, 1, 1),  initializer=RandomNormal(mean=1.0, stddev=0.02), trainable=True, name='freq_scale', 
                                              constraint=tf.keras.constraints.MaxNorm(2.0))
            self.spat_scale = self.add_weight(shape=(1, 1, 1, 1),  initializer=HeNormal(), trainable=True, name='spat_scale', constraint=tf.keras.constraints.MaxNorm(2.0))
            

        super(DeeperGlobalLocalAttentionLayerWithFrequency, self).build(input_shapes)

    def call(self, inputs, training=None):
        inputs1, inputs2 = inputs

        ######### Frequency Domain Transform #########
        freq1_dct = self.dct_transform(inputs1)
        freq1_fft = self.fft_transform(inputs2)
        

        ######### Frequency Features Attention #########
        freq1_dct_avg = self.global_avg_pooling(freq1_dct)
        freq1_dct_max = self.global_max_pooling(freq1_dct)
        freq1_dct_min = self.global_min_pooling(freq1_dct)

        freq1_fft_avg = self.global_avg_pooling(freq1_fft)
        freq1_fft_max = self.global_max_pooling(freq1_fft)
        freq1_fft_min = self.global_min_pooling(freq1_fft)
        
        ######### Spatial Domain Attention #########
        global_min = self.global_min_pooling1(inputs1)
        global_avg = self.global_avg_pooling1(inputs1)
        global_max = self.global_max_pooling1(inputs1)

        global_min1 = self.global_min_pooling1(inputs2)
        global_avg1 = self.global_avg_pooling1(inputs2)
        global_max1 = self.global_max_pooling1(inputs2)


        freq_avg_add = (freq1_dct_avg + freq1_fft_avg + global_avg + global_avg1) #* self.freq_scale
        freq_avg_sub = ((freq1_dct_avg + freq1_fft_avg) - (global_avg + global_avg1)) #* self.freq_scale

        freq_max_add = (freq1_dct_max + freq1_fft_max + global_max + global_max1) #* self.freq_scale
        freq_max_sub = ((freq1_dct_max + freq1_fft_max) - (global_max + global_max1)) #* self.freq_scale

        freq_min_add = (freq1_dct_min + freq1_fft_min + global_min + global_min1) #* self.freq_scale
        freq_min_sub = ((freq1_dct_min + freq1_fft_min) - (global_min + global_min1)) #* self.freq_scale

        freq_add = L.Activation("relu")((freq_avg_add + freq_max_add + freq_min_add)) 
        freq_sub = L.Activation("relu")((freq_avg_sub + freq_max_sub + freq_min_sub)) 

        freq_add = self.dropout(freq_add, training=training)
        freq_sub = self.dropout(freq_sub, training=training)
        
        #freq_add = tf.keras.layers.Dropout(0.3)(freq_add)
        #freq_sub = tf.keras.layers.Dropout(0.3)(freq_sub)

        #freq_spat_attention = L.Activation("sigmoid")(freq_add + freq_sub)
        #freq_spat_attention = tf.keras.layers.Dropout(0.2)(freq_spat_attention)
        
        #freq_spat_attention = tf.concat([freq_avg_add, freq_max_add, freq_min_add, freq_avg_sub, freq_max_sub, freq_min_sub], axis=-1)

        
        #freq_spat_attention = self.global_attention_freq(freq_spat_attention)
        freq_add = tf.expand_dims(tf.expand_dims(freq_add, 1), 1)
        freq_add = self.batch(freq_add)
        
        #freq_spat_attention = L.Activation("relu")(freq_spat_attention)
        
        print('freq_add:', freq_add.shape)

        freq_sub = tf.expand_dims(tf.expand_dims(freq_sub, 1), 1)
        freq_sub = self.batch(freq_sub)
        print('freq_sub:', freq_sub.shape)

        
        ####### Local attention ##########
        local1 = self.local_attn(inputs1)
        local2 = self.local_attn(inputs2)
        
        local_attention = tf.sigmoid(local1 + local2)
        
        
        ######### Combine Frequency and Spatial Attention #########
        if self.use_scale:
            freq_add *= self.global_scale
            freq_sub *= self.global_scale2
            local_attention *= self.local_scale
            
            #freq_attention *= self.global_scale
            #spat_attention *= self.global_scale2

        #local_attention1 = inputs1 * local_attention * self.local_scale
        #local_attention2 = inputs2 * local_attention * self.local_scale

        frequency = tf.sigmoid(freq_add + freq_sub)
        #attention = tf.sigmoid(freq_add + freq_sub + local_attention)
        
        attention = tf.sigmoid(frequency + local_attention)
        
        attention = attention / tf.reduce_sum(attention, axis=-1, keepdims=True)  # Normalize
        #combined_attention2 = tf.sigmoid(freq_spat_attention + freq_attention)
        
        return attention

    def get_config(self):
        config = super(DeeperGlobalLocalAttentionLayerWithFrequency, self).get_config()
        config.update({'units': self.units, 'activation': self.activation, 'dropout_rate': self.dropout_rate,
                       'use_scale': self.use_scale})
        return config

In [ ]:
# ================= PATCHED: DeeperGlobalLocalAttentionLayer1 (fix mkgc1 + typo) ================
import tensorflow as tf
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.signal import fft2d, dct
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, Input


import math
import tensorflow as tf
from tensorflow.keras import layers as L

import math
import tensorflow as tf
from tensorflow.keras import layers as L

class GhostPointwise(L.Layer):
    """
    Ghost pointwise (GhostNet-style 1x1 replacement).
    Out = concat( Conv1x1(x, m), DW3x3(Conv1x1(x, m)) )[:,:,:,out_channels]
    where m = ceil(out_channels / ratio).
    Greatly reduces 1x1 MACs while preserving accuracy.
    """
    def __init__(self, out_channels, ratio=2, use_bias=False, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = int(out_channels)
        self.ratio = int(max(2, ratio))
        self.use_bias = bool(use_bias)

        # sub-layers created in build
        self.pw = None
        self.pw_bn = None
        self.pw_act = None
        self.dw = None
        self.dw_bn = None
        self.dw_act = None

    def build(self, input_shape):
        m = int(math.ceil(self.out_channels / float(self.ratio)))
        # primary 1x1
        self.pw     = L.Conv2D(m, 1, padding="same", use_bias=self.use_bias, name=f"{self.name}_pw")
        self.pw_bn  = L.BatchNormalization(name=f"{self.name}_pw_bn")
        self.pw_act = L.Activation("relu", name=f"{self.name}_pw_act")
        # cheap depthwise features
        self.dw     = L.DepthwiseConv2D(3, padding="same",
                                        depth_multiplier=max(1, self.ratio - 1),
                                        use_bias=False, name=f"{self.name}_dw")
        self.dw_bn  = L.BatchNormalization(name=f"{self.name}_dw_bn")
        self.dw_act = L.Activation("relu", name=f"{self.name}_dw_act")
        super().build(input_shape)

    
    def call(self, x, training=None):
        p = self.pw(x); p = self.pw_bn(p, training=training); p = self.pw_act(p)
        d = self.dw(p); d = self.dw_bn(d, training=training); d = self.dw_act(d)
        y = tf.concat([p, d], axis=-1)
        return y[..., :self.out_channels]


class MultiKernelGroupwiseConv1(L.Layer):
    """
    Compact Multi‐Kernel Grouped Convolution (GFLOPs-reduced):
    - Four DW branches -> weighted sum (no 4× concat)
    - Depthwise 3x3 mix
    - Projection via **Ghost pointwise** (default) or grouped 1x1 (fallback)
    - Residual (identity or cheap projection)
    - Optional SE
    """
    def __init__(self, filters, groups=16, strides=1,
                 use_5x5=False, se_ratio=0.0, auto_groups=True,
                 proj_mode="ghost",       # "ghost" (recommended) or "grouped"
                 ghost_ratio=2,           # 2 is strong; 3 is lighter/cheaper
                 **kwargs):
        super().__init__(**kwargs)
        self.filters     = int(filters)
        self.groups      = int(groups)
        self.strides     = int(strides)
        self.use_5x5     = bool(use_5x5)
        self.se_ratio    = float(se_ratio)
        self.auto_groups = bool(auto_groups)
        self.proj_mode   = str(proj_mode)
        self.ghost_ratio = int(ghost_ratio)

        # parallel branches (DW convs)
        self.dw1x1 = L.DepthwiseConv2D(1, padding="same")
        self.dw3x3 = L.DepthwiseConv2D(3, padding="same")
        self.dw5x5 = L.DepthwiseConv2D(5, padding="same") if self.use_5x5 else None
        self.dw3d2 = L.DepthwiseConv2D(3, dilation_rate=2, padding="same")

        # mix + projection (created in build)
        self.dw_mix     = L.DepthwiseConv2D(3, padding="same")
        self.proj       = None  # GhostPointwise or grouped Conv2D(1x1)
        self.short_proj = None  # residual projection when needed

        # SE (optional)
        self.se_gap = None; self.se_fc1 = None; self.se_fc2 = None

        # branch weights
        self.a1 = None; self.a2 = None; self.a3 = None; self.a4 = None

        # light norm/act
        self.bn  = L.BatchNormalization()
        self.act = L.Activation("gelu")
        self.add = L.Add()

    def _make_grouped_1x1(self, in_ch, out_ch, name):
        # pick the largest valid groups (minimizes 1x1 cost)
        g_max = math.gcd(in_ch, out_ch)
        g_eff = g_max if self.auto_groups else max(1, min(self.groups, g_max))
        return L.Conv2D(out_ch, 1, padding="same", groups=max(1, g_eff), name=name)

    def build(self, input_shape):
        C_in = int(input_shape[-1])

        # learnable branch weights
        self.a1 = self.add_weight(shape=(), initializer="ones",  trainable=True, name=f"{self.name}_a1")
        self.a2 = self.add_weight(shape=(), initializer="ones",  trainable=True, name=f"{self.name}_a2")
        if self.use_5x5:
            self.a3 = self.add_weight(shape=(), initializer="zeros", trainable=True, name=f"{self.name}_a3")
        else:
            self.a3 = None
        self.a4 = self.add_weight(shape=(), initializer="ones",  trainable=True, name=f"{self.name}_a4")

        # projection choice
        if self.proj_mode.lower() == "ghost":
            self.proj = GhostPointwise(self.filters, ratio=self.ghost_ratio, name=f"{self.name}_gpw")
        else:
            self.proj = self._make_grouped_1x1(C_in, self.filters, name=f"{self.name}_grp1x1")

        # residual projection if needed
        if self.strides != 1 or C_in != self.filters:
            if self.proj_mode.lower() == "ghost":
                self.short_proj = GhostPointwise(self.filters, ratio=self.ghost_ratio, name=f"{self.name}_sc_gpw")
            else:
                self.short_proj = self._make_grouped_1x1(C_in, self.filters, name=f"{self.name}_sc1x1")
        else:
            self.short_proj = None

        # optional SE
        if self.se_ratio and self.se_ratio > 0.0:
            r = max(1, int(self.filters * self.se_ratio))
            self.se_gap = L.GlobalAveragePooling2D()
            self.se_fc1 = L.Dense(r, activation="relu")
            self.se_fc2 = L.Dense(self.filters, activation="sigmoid")

        super().build(input_shape)

    
    def call(self, x, training=None):
        # parallel DW branches (cheap)
        b1 = self.dw1x1(x)
        b2 = self.dw3x3(x)
        b4 = self.dw3d2(x)
        branches = [self.a1 * b1, self.a2 * b2, self.a4 * b4]
        if self.use_5x5:
            b3 = self.dw5x5(x)
            branches.insert(2, self.a3 * b3)

        # weighted sum (keeps channels = C_in)
        x1 = tf.add_n(branches)

        # DW mix + projection
        x1 = self.dw_mix(x1)
        x1 = self.proj(x1, training=training)

        # optional SE
        if self.se_ratio and self.se_ratio > 0.0:
            s = self.se_gap(x1)
            s = self.se_fc1(s)
            s = self.se_fc2(s)
            s = tf.reshape(s, [-1, 1, 1, self.filters])
            x1 = x1 * s

        # residual
        sc = x if self.short_proj is None else self.short_proj(x, training=training)
        out = self.add([sc, x1])
        out = self.bn(out, training=training)
        return self.act(out)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "filters": self.filters,
            "groups": self.groups,
            "strides": self.strides,
            "use_5x5": self.use_5x5,
            "se_ratio": self.se_ratio,
            "auto_groups": self.auto_groups,
            "proj_mode": self.proj_mode,
            "ghost_ratio": self.ghost_ratio,
        })
        return cfg


# ---- Drop-in alias (optional): uncomment to replace your class name globally ----
# MultiKernelGroupwiseConv1 = MultiKernelGroupwiseConv1Compact



class GlobalSumPooling2D(layers.Layer):
    """Global sum pooling over spatial dimensions."""
    def call(self, inputs):
        return tf.reduce_sum(inputs, axis=[1, 2])
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])
    def get_config(self):
        return super().get_config()


class GlobalMinPooling2D(layers.Layer):
    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2])
    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])
    def get_config(self):
        return super().get_config()


class DeeperGlobalLocalAttentionLayer1(layers.Layer):
    def __init__(self, units, activation='sigmoid', dropout_rate=0.2, use_scale=True, axis=-1, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_scale = use_scale
        self.axis = axis
        self.mkgc1 = None  # will be created in build()

    def build(self, input_shape):
        # Multi-kernel conv block used repeatedly
        #self.mkgc1 = MultiKernelGroupwiseConv1(filters=self.units, groups=16, strides=1)

        self.mkgc1 = MultiKernelGroupwiseConv1(
        filters=self.units,
        #groups=16,          # optional cap; set higher to allow more grouping if divisible
        strides=1,
        use_5x5=False,      # keep False for speed; set True if you really need it
        se_ratio=0.0,       # e.g., 0.0625 (~1/16) if you want tiny SE for quality
        auto_groups=True    # let it pick the largest valid groups automatically
        )
        


        # global stats pooling
        self.global_avg_pooling = layers.GlobalAveragePooling2D()
        self.global_max_pooling = layers.GlobalMaxPooling2D()
        self.global_min_pooling = GlobalMinPooling2D()
        self.global_sum_pooling = GlobalSumPooling2D()

        # extra (defined but not strictly needed for correctness)
        self.global_avg_pooling3 = layers.GlobalAveragePooling2D()
        self.global_max_pooling4 = layers.GlobalMaxPooling2D()
        self.global_min_pooling_4 = GlobalMinPooling2D()
        self.global_sum_pooling5 = GlobalSumPooling2D()

        # simple combiners
        self.add_op = layers.Add()
        self.cat_op = layers.Concatenate(axis=-1)

        # projection from global vector to channel mask
        self.global_attention = layers.Dense(units=self.units, activation=self.activation)

        # local/frequency convs
        self.local_conv1 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), groups=16, activation=self.activation)
        self.local_conv2 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), groups=16, activation=self.activation)

        if self.use_scale:
            self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale')
            self.local_scale  = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale')

        super().build(input_shape)

    
    def call(self, inputs, training=None):
        # ----- Global hierarchical information (via MKGC) -----
        ga1 = self.mkgc1(inputs)   # [B,H,W,C]
        avg1 = self.global_avg_pooling(ga1)
        max1 = self.global_max_pooling(ga1)
        min1 = self.global_min_pooling(ga1)
        sum1 = self.global_sum_pooling(ga1)
        add1 = self.add_op([avg1, max1, min1, sum1])
        sub1 = max1 - avg1 - min1
        g1   = add1 + sub1                                           # [B,C]

        ga2 = self.mkgc1(ga1)
        avg2 = self.global_avg_pooling(ga2)
        max2 = self.global_max_pooling(ga2)
        min2 = self.global_min_pooling(ga2)
        sum2 = self.global_sum_pooling(ga2)
        add2 = self.add_op([avg2, max2, min2, sum2])
        sub2 = max2 - avg2 - min2
        g2   = add2 + sub2

        ga3 = self.mkgc1(ga2)
        avg3 = self.global_avg_pooling(ga3)
        max3 = self.global_max_pooling(ga3)
        min3 = self.global_min_pooling(ga3)
        sum3 = self.global_sum_pooling(ga3)
        add3 = self.add_op([avg3, max3, min3, sum3])
        sub3 = max3 - avg3 - min3
        g3   = add3 + sub3

        # FIX: use ga3 (not a non-existent global_attention_3)
        ga_concat1 = self.add_op([ga1, ga2, ga3])                     # shape like ga1
        ga_sub_1   = ga2 - ga1 - ga3
        ga_concat1 = ga_concat1 + ga_sub_1

        ga4 = self.mkgc1(ga_concat1)
        avg4 = self.global_avg_pooling(ga4)
        max4 = self.global_max_pooling(ga4)
        min4 = self.global_min_pooling(ga4)
        sum4 = self.global_sum_pooling(ga4)
        add4 = self.add_op([avg4, max4, min4, sum4])
        sub4 = max4 - avg4 - min4
        g4   = add4 + sub4

        ga5 = self.mkgc1(ga4)
        avg5 = self.global_avg_pooling(ga5)
        max5 = self.global_max_pooling(ga5)
        min5 = self.global_min_pooling(ga5)
        sum5 = self.global_sum_pooling(ga5)
        add5 = self.add_op([avg5, max5, min5, sum5])
        sub5 = max5 - avg5 - min5
        g5   = add5 + sub5

        ga6 = self.mkgc1(ga5)
        avg6 = self.global_avg_pooling(ga6)
        max6 = self.global_max_pooling(ga6)
        min6 = self.global_min_pooling(ga6)
        sum6 = self.global_sum_pooling(ga6)
        add6 = self.add_op([avg6, max6, min6, sum6])
        sub6 = max6 - avg6 - min6
        g6   = add6 + sub6

        # Project global descriptors to channel weights
        g_vec = self.cat_op([g3, g6])                                 # [B, 2C]
        g_att = self.global_attention(g_vec)                          # [B, C]
        g_att = tf.expand_dims(tf.expand_dims(g_att, 1), 1)           # [B,1,1,C]

        # ----- Local / frequency attention -----
        shp = tf.shape(inputs)
        flat = tf.reshape(inputs, [-1, shp[-1]])                      # [B*H*W, C]
        dct  = tf.signal.dct(flat, type=2, norm='ortho')
        dct  = tf.reshape(dct, shp)                                   # [B,H,W,C]

        loc1 = self.local_conv1(dct)
        l1_max = tf.reduce_max(loc1, axis=[1,2], keepdims=True)
        l1_avg = tf.reduce_mean(loc1, axis=[1,2], keepdims=True)
        l1_min = tf.reduce_min(loc1, axis=[1,2], keepdims=True)
        l1_sum = tf.reduce_sum(loc1, axis=[1,2], keepdims=True)
        l1_add = self.add_op([l1_max, l1_avg, l1_min, l1_sum])

        loc2 = self.local_conv2(l1_add)
        l2_max = tf.reduce_max(loc2, axis=[1,2], keepdims=True)
        l2_avg = tf.reduce_mean(loc2, axis=[1,2], keepdims=True)
        l2_min = tf.reduce_min(loc2, axis=[1,2], keepdims=True)
        l2_sum = tf.reduce_sum(loc2, axis=[1,2], keepdims=True)

        loc3 = self.local_conv1(inputs)
        l3_max = tf.reduce_max(loc3, axis=[1,2], keepdims=True)
        l3_avg = tf.reduce_mean(loc3, axis=[1,2], keepdims=True)
        l3_min = tf.reduce_min(loc3, axis=[1,2], keepdims=True)
        l3_sum = tf.reduce_sum(loc3, axis=[1,2], keepdims=True)
        l3_add = self.add_op([l3_max, l3_avg, l3_min, l3_sum])

        loc4 = self.local_conv2(l3_add)
        l4_max = tf.reduce_max(loc4, axis=[1,2], keepdims=True)
        l4_avg = tf.reduce_mean(loc4, axis=[1,2], keepdims=True)
        l4_min = tf.reduce_min(loc4, axis=[1,2], keepdims=True)
        l4_sum = tf.reduce_sum(loc4, axis=[1,2], keepdims=True)

        l_max = self.add_op([l1_max, l2_max, l3_max, l4_max])
        l_avg = self.add_op([l1_avg, l2_avg, l3_avg, l4_avg])
        l_min = self.add_op([l1_min, l2_min, l3_min, l4_min])
        l_sum = self.add_op([l1_sum, l2_sum, l3_sum, l4_sum])

        l_att = tf.nn.relu(self.add_op([l_avg, l_max, l_min, l_sum]))  # [B,1,1,C]

        # scales
        if self.use_scale:
            g_att *= self.global_scale
            l_att *= self.local_scale

        att = tf.sigmoid(g_att + l_att)                                # [B,1,1,C]
        return att
# ================================================================================================


### Code started ###

In [ ]:
import math
import tensorflow as tf
import tensorflow.keras.layers as L

# ---------- cheap 1x1 replacement ----------
class GhostPointwise(L.Layer):
    def __init__(self, out_channels, ratio=2, use_bias=False, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = int(out_channels)
        self.ratio = int(max(2, ratio))
        self.use_bias = bool(use_bias)
        self.pw=None; self.pw_bn=None; self.pw_act=None
        self.dw=None; self.dw_bn=None; self.dw_act=None

    def build(self, input_shape):
        m = int(math.ceil(self.out_channels / float(self.ratio)))
        self.pw     = L.Conv2D(m, 1, padding="same", use_bias=self.use_bias, name=f"{self.name}_pw")
        self.pw_bn  = L.BatchNormalization(name=f"{self.name}_pw_bn")
        self.pw_act = L.Activation("relu", name=f"{self.name}_pw_act")
        self.dw     = L.DepthwiseConv2D(3, padding="same", depth_multiplier=max(1, self.ratio - 1),
                                        use_bias=False, name=f"{self.name}_dw")
        self.dw_bn  = L.BatchNormalization(name=f"{self.name}_dw_bn")
        self.dw_act = L.Activation("relu", name=f"{self.name}_dw_act")
        super().build(input_shape)

    
    def call(self, x, training=None):
        p = self.pw(x); p = self.pw_bn(p, training=training); p = self.pw_act(p)
        d = self.dw(p); d = self.dw_bn(d, training=training); d = self.dw_act(d)
        y = tf.concat([p, d], axis=-1)
        return y[..., :self.out_channels]

# ---------- tiny, shape‑preserving weighted fuse ----------
class WeightedCombinationSoft(L.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.logits = None
        self.n = None
    def build(self, input_shapes):
        self.n = len(input_shapes)
        self.logits = self.add_weight(shape=(self.n,), initializer="zeros",
                                      trainable=True, name=f"{self.name}_logits")
        super().build(input_shapes)
    
    def call(self, xs):
        # all xs must have same shape; we keep fusions at proj_c to ensure that
        w = tf.nn.softmax(self.logits)  # [n]
        out = 0.0
        for i, x in enumerate(xs):
            out = out + w[i] * x
        return out
    def compute_output_shape(self, input_shapes):
        return input_shapes[0]


'''class LowRankDense(layers.Layer):
    def __init__(self, units, rank, use_bias=True, **kw):
        super().__init__(**kw)
        self.units, self.rank, self.use_bias = units, rank, use_bias
    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        self.U = self.add_weight(name="U", shape=(in_dim, self.rank),
                                 initializer="glorot_uniform")
        self.V = self.add_weight(name="V", shape=(self.units, self.rank),
                                 initializer="glorot_uniform")
        if self.use_bias:
            self.b = self.add_weight(name="b", shape=(self.units,),
                                     initializer="zeros")
    def call(self, x):
        #  x @ U @ Vᵀ
        z = tf.linalg.matmul(x, self.U)
        z = tf.linalg.matmul(z, self.V, transpose_b=True)
        if self.use_bias:
            z = z + self.b
        return z'''


import tensorflow as tf
from tensorflow.keras import layers

class LowRankDense(layers.Layer):
    """
    Dense(x, units) ≈ x @ U @ V^T + b   with an in-layer activation.
    - rank << min(in_dim, units) reduces MACs from O(in*units) to O(in*rank + rank*units)
    - activation: "relu", "sigmoid", "hard_sigmoid", or None
    """
    def __init__(self, units, rank, activation=None, use_bias=True, **kw):
        super().__init__(**kw)
        self.units = int(units)
        self.rank = int(rank)
        self.activation = activation  # "relu" | "sigmoid" | "hard_sigmoid" | None
        self.use_bias = bool(use_bias)

        # Will be set in build
        self.U = None
        self.V = None
        self.b = None

    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        # x @ U @ V^T  (keep shapes consistent with your original code)
        self.U = self.add_weight(
            name="U", shape=(in_dim, self.rank),
            initializer="glorot_uniform", trainable=True
        )
        # original used (units, rank) with transpose_b=True
        self.V = self.add_weight(
            name="V", shape=(self.units, self.rank),
            initializer="glorot_uniform", trainable=True
        )
        if self.use_bias:
            self.b = self.add_weight(
                name="b", shape=(self.units,),
                initializer="zeros", trainable=True
            )
        super().build(input_shape)

    def call(self, x):
        #  x @ U @ Vᵀ
        z = tf.linalg.matmul(x, self.U)                     # [B, ..., rank]
        z = tf.linalg.matmul(z, self.V, transpose_b=True)   # [B, ..., units]
        if self.use_bias:
            z = tf.nn.bias_add(z, self.b)

        # Inline activation (keeps it a drop-in Dense replacement)
        if self.activation is None:
            return z
        if self.activation == "relu":
            return tf.nn.relu(z)
        if self.activation == "sigmoid":
            return tf.nn.sigmoid(z)
        if self.activation == "hard_sigmoid":
            # Cheaper than full sigmoid; good for gates
            return tf.keras.activations.hard_sigmoid(z)
        # fallback: allow any Keras activation string/callable
        act = tf.keras.activations.get(self.activation)
        return act(z)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            "units": self.units,
            "rank": self.rank,
            "activation": self.activation,
            "use_bias": self.use_bias,
        })
        return cfg

    def compute_output_shape(self, input_shape):
        return input_shape[:-1] + (self.units,)



# ---------- compact fusion with latent (proj_c) fusions ----------
def fusion_tasks(conv1_2, conv1_3, conv1_4, conv1_5,
                 proj_c=16,         # latent channels for fusion
                 fuse_c=64,         # output channels per stage (same as your original)
                 ghost_ratio=2,
                 use_se=False,
                 name_prefix="fuse"):
    gpw = lambda c, n: GhostPointwise(c, ratio=ghost_ratio, name=n)

    # 1) project at native resolution (cheap)
    p2 = gpw(proj_c, f"{name_prefix}_pre_p2")(conv1_2)
    p3 = gpw(proj_c, f"{name_prefix}_pre_p3")(conv1_3)
    p4 = gpw(proj_c, f"{name_prefix}_pre_p4")(conv1_4)
    p5 = gpw(proj_c, f"{name_prefix}_pre_p5")(conv1_5)

    # 2) upsample to full res (all have proj_c channels)
    U = lambda t, tag: L.UpSampling2D((2,2), interpolation="bilinear", name=f"{name_prefix}_up_{tag}")(t)
    u1 = U(U(U(p2, "p2a"), "p2b"), "p2c")   # ×8
    u2 = U(U(p3, "p3a"), "p3b")             # ×4
    u3 = U(p4, "p4")                         # ×2
    u4 = p5                                  # ×1

    # optional tiny anti‑alias
    def aa(x, tag): return L.DepthwiseConv2D(3, padding="same", name=f"{name_prefix}_aa_{tag}")(x)
    u1 = aa(u1, "u1"); u2 = aa(u2, "u2"); u3 = aa(u3, "u3")

    Fuse = WeightedCombinationSoft

    # ----- Stage 1: fuse in proj_c, then make 64‑ch output -----
    l1 = Fuse(name=f"{name_prefix}_w1")([u1, u2, u3, u4])     # [B,H,W,proj_c]
    o1 = gpw(fuse_c, f"{name_prefix}_o1")(l1)                 # [B,H,W,fuse_c]

    # ----- Stage 2: fuse l1 with others in proj_c -----
    l2 = Fuse(name=f"{name_prefix}_w2")([l1, u2, u3, u4])     # [B,H,W,proj_c]
    o2 = gpw(fuse_c, f"{name_prefix}_o2")(l2)

    # ----- Stage 3: fuse l1,l2 with others in proj_c -----
    l3 = Fuse(name=f"{name_prefix}_w3")([l1, l2, u3, u4])     # [B,H,W,proj_c]
    o3 = gpw(fuse_c, f"{name_prefix}_o3")(l3)

    # ----- Stage 4: fuse l1,l2,l3 with u4 in proj_c -----
    l4 = Fuse(name=f"{name_prefix}_w4")([l1, l2, l3, u4])     # [B,H,W,proj_c]
    o4 = gpw(fuse_c, f"{name_prefix}_o4")(l4)

    if use_se:
        def se_block(x, name):
            c = int(x.shape[-1]); r = max(1, c // 16)
            s = L.GlobalAveragePooling2D(name=f"{name}_gap")(x)
            
            #s = L.Dense(r, activation="relu", name=f"{name}_fc1")(s)
            

            
            s = LowRankDense(r, rank=max(1, r//4), activation="relu", use_bias=False)
            
            s = L.Dense(c, activation="sigmoid", name=f"{name}_fc2")(s)
            s = L.Reshape((1,1,c), name=f"{name}_rs")(s)
            return L.Multiply(name=f"{name}_mul")([x, s])
        o1 = se_block(o1, f"{name_prefix}_se1")
        o2 = se_block(o2, f"{name_prefix}_se2")
        o3 = se_block(o3, f"{name_prefix}_se3")
        o4 = se_block(o4, f"{name_prefix}_se4")

    return o1, o2, o3, o4


In [ ]:
# ================= OOM-SAFE MKGC (NO GHOST) + FIXED DEEPER ATTN ==================
import math
import tensorflow as tf
from tensorflow.keras import layers as L

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, Input


class FrequencyTransformLayer(layers.Layer):
    def __init__(self, transform_type='dct', **kwargs):
        super(FrequencyTransformLayer, self).__init__(**kwargs)
        self.transform_type = transform_type

    def call(self, inputs):
        # Ensure inputs are float32 for DCT and FFT operations
        inputs = tf.cast(inputs, tf.float32)

        if self.transform_type == 'dct':
            # Apply 2D DCT along the last axis
            input_shape = tf.shape(inputs)
            flattened_inputs = tf.reshape(inputs, [-1, input_shape[-1]])  # Flatten along the last axis
            dct_transformed = tf.signal.dct(flattened_inputs, type=2, norm='ortho')
            return tf.reshape(dct_transformed, input_shape)  # Reshape back to original dimensions
        elif self.transform_type == 'fft':
            # Apply 2D FFT and return the magnitude
            fft_transformed = tf.signal.fft2d(tf.cast(inputs, tf.complex64))
            return tf.math.abs(fft_transformed)
        else:
            raise ValueError("Unsupported transform type. Choose 'dct' or 'fft'.")

    def get_config(self):
        config = super(FrequencyTransformLayer, self).get_config()
        config.update({'transform_type': self.transform_type})
        return config


class MultiKernelGroupwiseConv1(layers.Layer):
    """
    Multi‐Kernel Grouped Convolution:
    - Four depthwise convs in parallel (1×1, 3×3, 5×5, 3×3 dilated)
    - Channel shuffle
    - Depthwise + grouped 1×1 conv
    - Downsampling with depthwise conv
    - Residual connection with matching shortcut
    """
    def __init__(self, filters, groups=8, strides=2, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.groups = groups
        self.strides = strides

        # parallel depthwise convs
        self.conv1x1      = layers.DepthwiseConv2D(1, padding="same")
        self.conv3x3      = layers.DepthwiseConv2D(3, padding="same")
        #self.conv5x5      = layers.DepthwiseConv2D(5, padding="same")
        #self.conv_dilated = layers.DepthwiseConv2D(3, dilation_rate=2, padding="same")

        # fusion + channel shuffle
        self.concat = layers.Concatenate()

        # hybrid convs
        #self.dw_conv       = layers.DepthwiseConv2D(3, padding="same")
        self.group_conv    = layers.Conv2D(filters, 1, groups=groups, padding="same")
        #self.dw_downsample = layers.DepthwiseConv2D(3, strides=strides, padding="same")

        # shortcut path
        self.short_conv1 = layers.Conv2D(filters, 1, groups=groups, padding="same")
        #self.short_dw    = layers.DepthwiseConv2D(3, strides=strides, padding="same")

        # final
        self.add = layers.Add()
        self.act = layers.Activation("gelu")

    def call(self, x):
        # parallel feature extraction
        p1 = self.conv1x1(x)
        p2 = self.conv3x3(x)
        #p3 = self.conv5x5(x)
        #p4 = self.conv_dilated(x)
        x1 = self.concat([p1, p2])

        '''# channel shuffle
        b, h, w, c = tf.unstack(tf.shape(x1))
        x1 = tf.reshape(x1, [b, h, w, self.groups, c // self.groups])
        x1 = tf.transpose(x1, [0, 1, 2, 4, 3])
        x1 = tf.reshape(x1, [b, h, w, c])'''

        # 2) Channel-shuffle using static C
        batch, H, W = tf.shape(x1)[0], tf.shape(x1)[1], tf.shape(x1)[2]
        C = x1.shape[-1]  # static channel dimension
        x1 = tf.reshape(x1, [batch, H, W, self.groups, C // self.groups])
        x1 = tf.transpose(x1, [0, 1, 2, 4, 3])
        x1 = tf.reshape(x1, [batch, H, W, C])


        # hybrid convolution + downsample
        #x1 = self.dw_conv(x1)
        x1 = self.group_conv(x1)
        #x1 = self.dw_downsample(x1)

        # shortcut
        shortcut = self.short_conv1(x)
        #shortcut = self.short_dw(shortcut)

        # merge & activate
        out = self.add([shortcut, x1])
        return self.act(out)

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "groups": self.groups,
            "strides": self.strides
        })
        return config


# Helper pooling layers used above
class GlobalSumPooling2D(L.Layer):
    def call(self, inputs): return tf.reduce_sum(inputs, axis=[1,2])
class GlobalMinPooling2D(L.Layer):
    def call(self, inputs): return tf.reduce_min(inputs, axis=[1,2])


# ---- DeeperGlobalLocalAttentionLayer1 that INSTantiates MKGC with GROUPED 1x1 ---
class DeeperGlobalLocalAttentionLayer1(tf.keras.layers.Layer):
    def __init__(self, units, activation='sigmoid', dropout_rate=0.2, use_scale=True, axis=-1, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_scale = use_scale
        self.axis = axis
        self.mkgc1 = None

    def build(self, input_shape):
        # Use the OOM-safe MKGC (grouped 1x1), no Ghost
        self.mkgc1 = MultiKernelGroupwiseConv1(filters=self.units, groups=8, strides=2)

        # global stats pooling
        self.global_avg_pooling = L.GlobalAveragePooling2D()
        self.global_max_pooling = L.GlobalMaxPooling2D()
        self.global_min_pooling = GlobalMinPooling2D()
        self.global_sum_pooling = GlobalSumPooling2D()

        self.add_op = L.Add()
        self.cat_op = L.Concatenate(axis=-1)
        
        #self.global_attention = L.Dense(units=self.units, activation=self.activation)

        self.global_attention = LowRankDense(self.units, rank=max(1, self.units//4), activation=self.activation, use_bias=False)

        # local 1x1 with SAFE groups (avoid invalid group counts)
        g_loc = max(1, math.gcd(self.units, 8))   # <=8 groups to keep kernels contiguous
        
        #self.local_conv1 = L.Conv2D(self.units, 1, padding="same", groups=g_loc, activation=self.activation)

        self.local_conv1 = GhostPointwise(self.units, ratio=2, use_bias=False, name=f"{self.name}_loc1_gpw")
        self.local_act1  = L.Activation(self.activation, name=f"{self.name}_loc1_act")

        self.local_conv2 = GhostPointwise(self.units, ratio=2, use_bias=False, name=f"{self.name}_loc2_gpw")
        self.local_act2  = L.Activation(self.activation, name=f"{self.name}_loc2_act")
        
        
        #self.local_conv2 = L.Conv2D(self.units, 1, padding="same", groups=g_loc, activation=self.activation)

        if self.use_scale:
            self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale')
            self.local_scale  = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale')

        super().build(input_shape)

    
    def call(self, inputs, training=None):
        # ----- Global hierarchical information (via MKGC) -----
        ga1 = self.mkgc1(inputs, training=training)
        avg1 = self.global_avg_pooling(ga1); max1 = self.global_max_pooling(ga1)
        min1 = self.global_min_pooling(ga1); sum1 = self.global_sum_pooling(ga1)
        g1   = self.add_op([self.add_op([avg1, max1, min1, sum1]), max1 - avg1 - min1]) ## Inter-module fusion

        ga2 = self.mkgc1(ga1, training=training)
        avg2 = self.global_avg_pooling(ga2); max2 = self.global_max_pooling(ga2)
        min2 = self.global_min_pooling(ga2); sum2 = self.global_sum_pooling(ga2)
        g2   = self.add_op([self.add_op([avg2, max2, min2, sum2]), max2 - avg2 - min2])

        ga3 = self.mkgc1(ga2, training=training)
        avg3 = self.global_avg_pooling(ga3); max3 = self.global_max_pooling(ga3)
        min3 = self.global_min_pooling(ga3); sum3 = self.global_sum_pooling(ga3)
        g3   = self.add_op([self.add_op([avg3, max3, min3, sum3]), max3 - avg3 - min3])

        ga_concat1 = self.add_op([ga1, ga2, ga3])
        ga_concat1 = ga_concat1 + (ga2 - ga1 - ga3)

        ga4 = self.mkgc1(ga_concat1, training=training)
        avg4 = self.global_avg_pooling(ga4); max4 = self.global_max_pooling(ga4)
        min4 = self.global_min_pooling(ga4); sum4 = self.global_sum_pooling(ga4)
        g4   = self.add_op([self.add_op([avg4, max4, min4, sum4]), max4 - avg4 - min4])

        ga5 = self.mkgc1(ga4, training=training)
        avg5 = self.global_avg_pooling(ga5); max5 = self.global_max_pooling(ga5)
        min5 = self.global_min_pooling(ga5); sum5 = self.global_sum_pooling(ga5)
        g5   = self.add_op([self.add_op([avg5, max5, min5, sum5]), max5 - avg5 - min5])

        ga6 = self.mkgc1(ga5, training=training)
        avg6 = self.global_avg_pooling(ga6); max6 = self.global_max_pooling(ga6)
        min6 = self.global_min_pooling(ga6); sum6 = self.global_sum_pooling(ga6)
        g6   = self.add_op([self.add_op([avg6, max6, min6, sum6]), max6 - avg6 - min6])

        #g_vec = self.cat_op([g3, g6])                 # [B, 2C]
        g_vec = self.cat_op([g1, g2, g3, g4, g5, g6])                 # [B, 2C]
        g_att = self.global_attention(g_vec)          # [B, C]
        g_att = g_att[:, tf.newaxis, tf.newaxis, :]   # [B,1,1,C]

        # ----- Local / frequency attention (unchanged) -----
        shp  = tf.shape(inputs)
        flat = tf.reshape(inputs, [-1, shp[-1]])
        dct  = tf.signal.dct(flat, type=2, norm='ortho')
        dct  = tf.reshape(dct, shp)

        #loc1 = self.local_conv1(dct, training=training)
        
        loc1_lin = self.local_conv1(dct, training=training)
        loc1     = self.local_act1(loc1_lin)

        l1_max = tf.reduce_max(loc1, axis=[1,2], keepdims=True)
        l1_avg = tf.reduce_mean(loc1, axis=[1,2], keepdims=True)
        l1_min = tf.reduce_min(loc1, axis=[1,2], keepdims=True)
        l1_sum = tf.reduce_sum(loc1, axis=[1,2], keepdims=True)
        l1_add = l1_max + l1_avg + l1_min + l1_sum

        #loc2 = self.local_conv2(l1_add, training=training)
        
        loc2_lin = self.local_conv2(l1_add, training=training)
        loc2     = self.local_act2(loc2_lin)
        
        l2_max = tf.reduce_max(loc2, axis=[1,2], keepdims=True)
        l2_avg = tf.reduce_mean(loc2, axis=[1,2], keepdims=True)
        l2_min = tf.reduce_min(loc2, axis=[1,2], keepdims=True)
        l2_sum = tf.reduce_sum(loc2, axis=[1,2], keepdims=True)

        #loc3 = self.local_conv1(inputs, training=training)

        loc3_lin = self.local_conv1(inputs, training=training)
        loc3     = self.local_act1(loc3_lin)
        
        l3_max = tf.reduce_max(loc3, axis=[1,2], keepdims=True)
        l3_avg = tf.reduce_mean(loc3, axis=[1,2], keepdims=True)
        l3_min = tf.reduce_min(loc3, axis=[1,2], keepdims=True)
        l3_sum = tf.reduce_sum(loc3, axis=[1,2], keepdims=True)
        l3_add = l3_max + l3_avg + l3_min + l3_sum

        #loc4 = self.local_conv2(l3_add, training=training)

        loc4_lin = self.local_conv2(l3_add, training=training)
        loc4     = self.local_act2(loc4_lin)

        
        l4_max = tf.reduce_max(loc4, axis=[1,2], keepdims=True)
        l4_avg = tf.reduce_mean(loc4, axis=[1,2], keepdims=True)
        l4_min = tf.reduce_min(loc4, axis=[1,2], keepdims=True)
        l4_sum = tf.reduce_sum(loc4, axis=[1,2], keepdims=True)

        l_max = l1_max + l2_max + l3_max + l4_max
        l_avg = l1_avg + l2_avg + l3_avg + l4_avg
        l_min = l1_min + l2_min + l3_min + l4_min
        l_sum = l1_sum + l2_sum + l3_sum + l4_sum

        l_att = tf.nn.relu(l_avg + l_max + l_min + l_sum)

        if self.use_scale:
            g_att *= self.global_scale
            l_att *= self.local_scale

        return tf.sigmoid(g_att + l_att)

# ================================================================================


In [ ]:
# ===================== Hyperbolic (Poincaré + Lorentz) Attention =====================
import tensorflow as tf
from tensorflow.keras import layers

class HyperbolicDualGeometry(layers.Layer):
    """
    Parallel hyperbolic attention (no quantum):
      - Computes channel-wise attention from two inputs using:
          (i) Lorentz term:  -t^2 + sum( (x^2) * beta )
         (ii) Poincaré term: (poincare(x) ⊙ x)
      - Combines them with learnable gates, produces [B,1,1,units] attention.

    Notes:
      * Uses global-average pooled features as channel descriptors (cheap, robust).
      * If channels != units, maps channel vector to 'units' via a Dense.
      * No layer/variable creation inside `call`.
    """
    def __init__(self, units, learnable_curvature=True, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.learnable_curvature = bool(learnable_curvature)

    def build(self, input_shapes):
        # Expect a pair: (x1, x2) = [B,H,W,C1], [B,H,W,C2]
        shape1, shape2 = input_shapes
        C1 = int(shape1[-1]); C2 = int(shape2[-1])
        # We'll operate channel-wise; unify them to a common size via small Dense if needed
        self.C = max(C1, C2)

        # Pooling to channel descriptors
        self.gap1 = layers.GlobalAveragePooling2D()
        self.gap2 = layers.GlobalAveragePooling2D()

        # Linear maps to common channel length C (if needed)
        
        #self.map1 = layers.Dense(self.C, use_bias=False) if C1 != self.C else None

        self.map1 = LowRankDense(self.C, rank=max(1, self.C//4), use_bias=False) if C1 != self.C else None
        
        #self.map2 = layers.Dense(self.C, use_bias=False) if C2 != self.C else None

        self.map2 = LowRankDense(self.C, rank=max(1, self.C//4), use_bias=False) if C2 != self.C else None

        # Optional projection to 'units' after attention computation
        
        #self.to_units = layers.Dense(self.units, use_bias=False) if self.C != self.units else None

        self.to_units = LowRankDense(self.units, rank=max(1, self.units//4), use_bias=False) if self.C != self.units else None

        # Curvature (positive): c = softplus(curv) + eps
        self.curv_log = self.add_weight(
            shape=(), initializer=tf.keras.initializers.Zeros(),
            trainable=self.learnable_curvature, name=f"{self.name}_curv_log"
        )

        # Per-channel scale for Lorentz spatial term
        self.beta = self.add_weight(
            shape=(self.C,), initializer="ones", trainable=True, name=f"{self.name}_beta"
        )

        # Gates for mixing the two geometry terms
        self.g_lorentz  = self.add_weight(shape=(), initializer="ones",   trainable=True, name=f"{self.name}_g_lorentz")
        self.g_poincare = self.add_weight(shape=(), initializer="ones",   trainable=True, name=f"{self.name}_g_poincare")

        super().build(input_shapes)

    
    def call(self, inputs, training=None):
        x1, x2 = inputs  # [B,H,W,C1], [B,H,W,C2]
        # Channel descriptors
        v1 = self.gap1(x1)  # [B, C1]
        v2 = self.gap2(x2)  # [B, C2]
        if self.map1 is not None: v1 = self.map1(v1)  # [B, C]
        if self.map2 is not None: v2 = self.map2(v2)  # [B, C]

        # Normalize (stabilize scales)
        def _znorm(v):
            m = tf.reduce_mean(v, axis=-1, keepdims=True)
            s = tf.math.reduce_std(v, axis=-1, keepdims=True) + 1e-6
            return (v - m) / s
        v1 = _znorm(v1); v2 = _znorm(v2)

        # Curvature
        c = tf.nn.softplus(self.curv_log) + 1e-3  # >0

        # ---------- Lorentz part ----------
        # t = sqrt(1 + c * ||v||^2) / c  (scalar per sample)
        n1 = tf.reduce_sum(tf.square(v1), axis=-1, keepdims=True)  # [B,1]
        n2 = tf.reduce_sum(tf.square(v2), axis=-1, keepdims=True)
        t1 = tf.math.sqrt(1.0 + c * n1) / c
        t2 = tf.math.sqrt(1.0 + c * n2) / c

        # Channel-wise Lorentz term:  -t^2 + (v^2 * beta)
        beta = tf.reshape(self.beta, [1, self.C])
        lor1 = -tf.square(t1) + tf.square(v1) * beta   # [B,C]
        lor2 = -tf.square(t2) + tf.square(v2) * beta   # [B,C]

        # ---------- Poincaré part ----------
        # poincare(v) = tanh(sqrt(c)*||v||) * v / ||v||
        def _poincare(v):
            n = tf.norm(v, axis=-1, keepdims=True) + 1e-6
            scale = tf.math.tanh(tf.sqrt(c) * n) / n
            return v * scale  # [B,C]
        p1 = _poincare(v1); p2 = _poincare(v2)

        # Channel-wise Poincaré term: elementwise product with original
        poi1 = p1 * v1  # [B,C]
        poi2 = p2 * v2  # [B,C]

        # ---------- Combine the two modalities ----------
        # We aggregate both inputs symmetrically then squash
        h = tf.nn.sigmoid(self.g_lorentz * (lor1 + lor2) + self.g_poincare * (poi1 + poi2))  # [B,C]

        # Map to 'units' if needed
        if self.to_units is not None:
            h = self.to_units(h)  # [B,units]

        # [B,1,1,units] attention mask
        h = tf.expand_dims(tf.expand_dims(h, axis=1), axis=1)
        return h
# ======================================================================================


In [ ]:
# ========== PATCH: DeeperGlobalLocalAttentionLayerWithFrequency ==========
class DeeperGlobalLocalAttentionLayerWithFrequency(layers.Layer):
    def __init__(self, units, activation='sigmoid', dropout_rate=0.3, use_scale=True, axis=-1, **kwargs):
        super(DeeperGlobalLocalAttentionLayerWithFrequency, self).__init__(**kwargs)
        self.units = units
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_scale = use_scale
        self.axis = axis

        self.dct_transform = FrequencyTransformLayer(transform_type='dct')
        self.fft_transform = FrequencyTransformLayer(transform_type='fft')

        self.global_avg_pooling = layers.GlobalAveragePooling2D()
        self.global_min_pooling = GlobalMinPooling2D()
        self.global_max_pooling = layers.GlobalMaxPooling2D()
        self.global_attention_freq = layers.Dense(units=self.units, activation=self.activation)
        self.local_attn = DeeperGlobalLocalAttentionLayer1(units=self.units, activation='sigmoid',
                                                           dropout_rate=0.2, use_scale=self.use_scale)
        # hyperbolic parallel branch
        self.hyper = None  # built later
        self.fuse_a = None # scalar mix for (freq/local) branch
        self.fuse_b = None # scalar mix for hyperbolic branch


    def build(self, input_shapes):
        input_shape1, input_shape2 = input_shapes

        self.global_min_pooling1 = GlobalMinPooling2D()
        self.global_avg_pooling1 = layers.GlobalAveragePooling2D()
        self.global_max_pooling1 = layers.GlobalMaxPooling2D()
        self.batch = BatchNormalization()
        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)

        #self.local_conv1 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)

        #self.local_conv3 = GhostPointwise(self.units, ratio=2, use_bias=False, name=f"{self.name}_loc3_gpw")
        #self.local_act3  = L.Activation(self.activation, name=f"{self.name}_loc3_act")


        if self.use_scale:
            self.global_scale  = self.add_weight(shape=(1, 1, 1, 1), initializer=tf.keras.initializers.RandomNormal(mean=1.0, stddev=0.02), trainable=True, name='global_scale', constraint=tf.keras.constraints.MaxNorm(2.0))
            self.global_scale2 = self.add_weight(shape=(1, 1, 1, 1), initializer=tf.keras.initializers.HeNormal(), trainable=True, name='global_scale2', constraint=tf.keras.constraints.MaxNorm(2.0))
            self.local_scale   = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale', constraint=tf.keras.constraints.MaxNorm(2.0))
            self.freq_scale    = self.add_weight(shape=(1, 1, 1, 1), initializer=tf.keras.initializers.RandomNormal(mean=1.0, stddev=0.02), trainable=True, name='freq_scale', constraint=tf.keras.constraints.MaxNorm(2.0))
            self.spat_scale    = self.add_weight(shape=(1, 1, 1, 1), initializer=tf.keras.initializers.HeNormal(), trainable=True, name='spat_scale', constraint=tf.keras.constraints.MaxNorm(2.0))

        # NEW: build hyperbolic branch and fusion scalars
        self.hyper = HyperbolicDualGeometry(units=self.units, learnable_curvature=True, name=f"{self.name}_hyper")
        self.hyper.build(input_shapes)  # explicit build to avoid graph var creation
        self.fuse_a = self.add_weight(shape=(), initializer="ones",   trainable=True, name=f"{self.name}_fuse_a")
        self.fuse_b = self.add_weight(shape=(), initializer="zeros",  trainable=True, name=f"{self.name}_fuse_b")

        super(DeeperGlobalLocalAttentionLayerWithFrequency, self).build(input_shapes)

    def call(self, inputs, training=None):
        inputs1, inputs2 = inputs

        # Frequency transforms
        freq1_dct = self.dct_transform(inputs1)
        freq1_fft = self.fft_transform(inputs2)

        # Frequency features
        dct_avg = self.global_avg_pooling(freq1_dct); dct_max = self.global_max_pooling(freq1_dct); dct_min = self.global_min_pooling(freq1_dct)
        fft_avg = self.global_avg_pooling(freq1_fft); fft_max = self.global_max_pooling(freq1_fft); fft_min = self.global_min_pooling(freq1_fft)

        # Spatial features
        gmin  = self.global_min_pooling1(inputs1); gavg  = self.global_avg_pooling1(inputs1); gmax  = self.global_max_pooling1(inputs1)
        gmin2 = self.global_min_pooling1(inputs2); gavg2 = self.global_avg_pooling1(inputs2); gmax2 = self.global_max_pooling1(inputs2)

        # Combine; use tf.nn.relu (no Keras Activation inside call)
        freq_avg_add = (dct_avg + fft_avg + gavg + gavg2)
        freq_avg_sub = (dct_avg + fft_avg) - (gavg + gavg2)
        freq_max_add = (dct_max + fft_max + gmax + gmax2)
        freq_max_sub = (dct_max + fft_max) - (gmax + gmax2)
        freq_min_add = (dct_min + fft_min + gmin + gmin2)
        freq_min_sub = (dct_min + fft_min) - (gmin + gmin2)

        freq_add = tf.nn.relu(freq_avg_add + freq_max_add + freq_min_add)
        freq_sub = tf.nn.relu(freq_avg_sub + freq_max_sub + freq_min_sub)

        freq_add = self.dropout(freq_add, training=training)
        freq_sub = self.dropout(freq_sub, training=training)

        freq_add = tf.expand_dims(tf.expand_dims(freq_add, 1), 1)
        freq_add = self.batch(freq_add, training=training)
        freq_sub = tf.expand_dims(tf.expand_dims(freq_sub, 1), 1)
        freq_sub = self.batch(freq_sub, training=training)

        # Local attentions (reusing sublayer; no variable creation now)
        local1 = self.local_attn(inputs1, training=training)
        local2 = self.local_attn(inputs2, training=training)
        local_attention = tf.sigmoid(local1 + local2)

        if self.use_scale:
            freq_add      *= self.global_scale
            freq_sub      *= self.global_scale2
            local_attention *= self.local_scale

        frequency = tf.sigmoid(freq_add + freq_sub)
        attention = tf.sigmoid(frequency + local_attention)

        '''# normalize safely
        denom = tf.reduce_sum(attention, axis=-1, keepdims=True) + 1e-6
        attention = attention / denom
        return attention'''

        # --------- NEW: hyperbolic branch (parallel)
        hyper_att = self.hyper([inputs1, inputs2], training=training)   # [B,1,1,units]

        # --------- Fuse the two branches (scalar mixing, then squash)
        fused = tf.sigmoid(self.fuse_a * attention + self.fuse_b * hyper_att)  # [B,1,1,units]

        

        # normalize safely across channels
        denom = tf.reduce_sum(fused, axis=-1, keepdims=True) + 1e-6
        fused = fused / denom
        return fused #attention
# =========================================================================

# =========================================================================


In [ ]:
class DeeperAttentionLayer(layers.Layer):
    def __init__(self, units=64, use_scale=True, axis=-1, scales=[1.0, 0.5, 0.25], **kwargs):
        """
        Initialize the DeeperAttentionLayer.
        Args:
            units: Number of units for attention layers.
            use_scale: Whether to scale attention values.
            axis: Axis for attention scaling.
            scales: List of scaling factors for multiscaling inputs.
        """
        super(DeeperAttentionLayer, self).__init__(**kwargs)
        self.units = units
        self.use_scale = use_scale
        self.axis = axis
        self.scales = scales

    def build(self, input_shapes):
        input_shape1, input_shape2 = input_shapes
        _, H, W, C1 = input_shape1
        _, H, W, C2 = input_shape2
        
        self.alpha1 = self.add_weight(shape=(1, 1, 1, C1), initializer='ones', trainable=True, name='alpha1')
        self.alpha2 = self.add_weight(shape=(1, 1, 1, C2), initializer='ones', trainable=True, name='alpha2')
        
        self.deeper_global_local_attention = DeeperGlobalLocalAttentionLayerWithFrequency(
            units=self.units,
            activation='sigmoid',
            dropout_rate=0.2,  # Adjust the dropout rate
            use_scale=self.use_scale
        )
        
        super(DeeperAttentionLayer, self).build(input_shapes)

    '''def generate_multiscaling_inputs(self, inputs):
        """
        Generate multiscaled versions of the input.
        Args:
            inputs: Original input tensor.
        Returns:
            List of multiscaled tensors.
        """
        multiscaled_inputs = []
        original_shape = tf.shape(inputs)[1:3]  # Height and Width
        for scale in self.scales:
            scaled_shape = tf.cast(original_shape * scale, tf.int32)
            scaled_input = tf.image.resize(inputs, scaled_shape, method='bilinear')
            scaled_input = tf.image.resize(scaled_input, original_shape, method='bilinear')  # Restore to original size
            multiscaled_inputs.append(scaled_input)
        return multiscaled_inputs'''

    def call(self, inputs, training=True):
        inputs1, inputs2 = inputs

        # Generate multiscaled inputs
        #scaled_inputs1 = self.generate_multiscaling_inputs(inputs1)
        #scaled_inputs2 = self.generate_multiscaling_inputs(inputs2)

        # Fuse multiscaled inputs with the original inputs (e.g., concatenate or add)
        #multiscaled_inputs1 = tf.add_n([inputs1] + scaled_inputs1)
        #multiscaled_inputs2 = tf.add_n([inputs2] + scaled_inputs2)

        # Apply attention
        attention = self.deeper_global_local_attention([inputs1, inputs2], training=training)
        #attention = self.deeper_global_local_attention([inputs1, inputs2])#, training=training)

        # Scale with alpha weights
        attention_feature1 = inputs1 * attention * self.alpha1
        attention_feature2 = inputs2 * attention * self.alpha2

        #return attention_feature1, attention_feature2
        #return inputs1 *  (attention_feature1 + attention_feature2)
        return attention_feature1, attention_feature2

    def get_config(self):
        config = super(DeeperAttentionLayer, self).get_config()
        config.update({'units': self.units, 'use_scale': self.use_scale, 'scales': self.scales})
        return config


In [ ]:
# ====================== GATED DEEPER ATTENTION (CONTROL-READY) ======================
class ConcreteGate(L.Layer):
    """
    Concrete (Gumbel–Sigmoid) gate with *controllable* penalty and temperature.
    The controller will adjust lmbda_var to meet a compute-time budget.
    """
    def __init__(self, temperature_init=2.0, lmbda_init=5e-5, **kwargs):
        super().__init__(**kwargs)
        self.temperature_init = float(temperature_init)
        self.lmbda_init = float(lmbda_init)

    def build(self, input_shape):
        # Stochastic gate parameter (trainable)
        self.logit = self.add_weight(
            shape=(), initializer=tf.keras.initializers.Constant(-2.0),
            trainable=True, name="gate_logit"
        )
        # Temperature & penalty are non-trainable but mutable (controller updates them)
        self.temperature_var = self.add_weight(
            shape=(), initializer=tf.keras.initializers.Constant(self.temperature_init),
            trainable=False, name="gate_temperature"
        )
        self.lmbda_var = self.add_weight(
            shape=(), initializer=tf.keras.initializers.Constant(self.lmbda_init),
            trainable=False, name="gate_lmbda"
        )

    def call(self, x, training=None):
        if training is None:
            training = False
        training = tf.cast(tf.convert_to_tensor(training), tf.bool)

        def _sample(temp):
            u = tf.random.uniform(())
            g = tf.math.log(u + 1e-8) - tf.math.log(1.0 - u + 1e-8)
            return tf.sigmoid((self.logit + g) / temp)  # scalar in (0,1)

        def _det(_temp_unused):
            return tf.sigmoid(self.logit)

        z = tf.cond(training, lambda: _sample(self.temperature_var), lambda: _det(self.temperature_var))
        # Controllable compute penalty (encourages skipping when training gets too slow)
        self.add_loss(self.lmbda_var * z)
        return tf.reshape(z, [1, 1, 1, 1])  # broadcast

class GatedDeeperAttentionLayer(L.Layer):
    """
    Wraps your DeeperAttentionLayer with a learnable gate and a cheap skip path.
    Returns a single fused tensor (same shape as inputs).
    """
    def __init__(self, units, temperature_init=2.0, lmbda_init=5e-5, **kwargs):
        super().__init__(**kwargs)
        self.units = int(units)
        self.inner = DeeperAttentionLayer(units=units, use_scale=True)
        self.gate  = ConcreteGate(temperature_init=temperature_init, lmbda_init=lmbda_init)

    def call(self, inputs, training=None):
        x, s = inputs
        a1, a2 = self.inner([x, s], training=training)  # heavy path
        heavy = a1 + a2
        skip  = x + s                                  # cheap path
        z = self.gate(x, training=training)            # [1,1,1,1]
        return z * heavy + (1.0 - z) * skip
# ====================================================================================



In [ ]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, Input, DepthwiseConv2D

from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50

from tensorflow.keras.initializers import RandomNormal, HeNormal
from tensorflow.keras.constraints import MaxNorm


# #lambda function for repeating the result from AG
def repeat_elem(tensor, rep):
   
     return tf.keras.layers.Lambda(lambda x, repnum: K.repeat_elements(x, repnum, axis=3),
                          arguments={'repnum': rep})(tensor)



# Attention Gate



'''def attention_gate(x, s, num_filters):
    
    #x1, s1 = DeeperAttentionLayer(units=num_filters, use_scale=True)([x, s])
    Wg = L.Conv2D(num_filters, 1, padding="same")(x)
    Wg = L.BatchNormalization()(Wg)

    Ws = L.Conv2D(num_filters, 1, padding="same")(s)
    Ws = L.BatchNormalization()(Ws)

    out = L.Activation("relu")(Wg + Ws)
    
    out = L.Conv2D(num_filters, 1, padding="same")(out)
    out = L.Activation("sigmoid")(out)

    return out * s'''




from tensorflow.keras import backend as K
import tensorflow as tf
import tensorflow.keras.layers as L

class ChannelScale(L.Layer):
    def __init__(self, init=0.5, **kwargs):
        super().__init__(**kwargs)
        self.init = float(init)
    def build(self, input_shape):
        C = int(input_shape[-1])
        self.gamma = self.add_weight(
            shape=(1, 1, 1, C),
            initializer=tf.keras.initializers.Constant(self.init),
            trainable=True,
            name=f"{self.name}_gamma"
        )
    def call(self, x):
        return x * self.gamma


class GhostHyperAttentionGate(L.Layer):
    """
    GhostNet-style attention gate + optional HyperbolicDualGeometry mixing.
    This version auto-uniques its own name if a base `name=` was reused.
    Internal sublayers get deterministic suffixes to stay unique per instance.
    """
    def __init__(self, num_filters=None, ratio=2, use_hyper=True, hyper_gain_init=0.5,
                 name=None, **kwargs):
        # Auto-unique the top-level layer name if a base name is reused
        if name is not None:
            name = f"{name}_{K.get_uid(name)}"
        super().__init__(name=name, **kwargs)

        self.num_filters     = None if num_filters is None else int(num_filters)
        self.ratio           = int(ratio)
        self.use_hyper       = bool(use_hyper)
        self.hyper_gain_init = float(hyper_gain_init)

        # Placeholders for sublayers
        self.gpw_x = None
        self.gpw_s = None
        self.gpw_out = None
        self.add_op = L.Add()
        self.relu   = L.Activation("relu")
        self.sigmoid= L.Activation("sigmoid")
        self.hyper  = None
        self.hyper_gain = None

    def build(self, input_shapes):
        shape_x, shape_s = input_shapes
        Cs = int(shape_s[-1])
        C  = self.num_filters if self.num_filters is not None else Cs
        if C is None:
            raise ValueError("GhostHyperAttentionGate: cannot infer num_filters; pass num_filters= explicitly.")

        # Use deterministic suffixes tied to this instance's unique self.name
        self.gpw_x   = GhostPointwise(C, ratio=self.ratio, use_bias=False, name=f"{self.name}_gpw_x")
        self.gpw_s   = GhostPointwise(C, ratio=self.ratio, use_bias=False, name=f"{self.name}_gpw_s")
        self.gpw_out = GhostPointwise(C, ratio=self.ratio, use_bias=False, name=f"{self.name}_gpw_out")

        if self.use_hyper:
            self.hyper      = HyperbolicDualGeometry(units=C, name=f"{self.name}_hyper")
            self.hyper_gain = ChannelScale(init=self.hyper_gain_init, name=f"{self.name}_hyper_gain")

        super().build(input_shapes)

    def call(self, inputs, training=None):
        x, s = inputs
        Wg = self.gpw_x(x, training=training)
        Ws = self.gpw_s(s, training=training)
        z_spat_in = self.add_op([Wg, Ws])
        z_spat_in = self.relu(z_spat_in)
        z_spat    = self.gpw_out(z_spat_in, training=training)

        if self.use_hyper:
            z_h = self.hyper([x, s], training=training)  # [B,1,1,C]
            z_h = self.hyper_gain(z_h)
            z_pre = self.add_op([z_spat, z_h])
        else:
            z_pre = z_spat

        z = self.sigmoid(z_pre)
        return z * s












# ====================== PATCH: attention_gate (safer activations) ======================
# =======================================================================================



# Convolution Block: Used in both encoder and decoder.

'''def conv_block(input, num_filters):
    #x = Conv2D(num_filters,3, padding="same")(input)
    x = DepthwiseConv2D(3, padding="same")(input)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    return x'''


import tensorflow as tf
import tensorflow.keras.layers as L

def conv_block(x, num_filters, ratio=2, se_ratio=0.0, name=None):
    """
    Ghost-conv version of your conv_block:
      DW3x3 -> BN -> ReLU ->
      DW3x3 -> GhostPointwise(1x1) -> (optional SE) -> BN -> ReLU

    Args:
      x            : input tensor
      num_filters  : output channels (same as before)
      ratio        : Ghost expansion ratio (2 is a good default; 3 is lighter)
      se_ratio     : 0.0 disables SE; e.g., 0.0625 (1/16) for tiny SE
      name         : optional base name to avoid collisions

    Returns:
      tensor with shape [B, H, W, num_filters]
    """
    nm = (lambda s: None if name is None else f"{name}_{s}")

    # Stage 1: same as before (cheap)
    x = L.DepthwiseConv2D(3, padding="same", name=nm("dw3a"))(x)
    x = L.BatchNormalization(name=nm("bn1"))(x)
    x = L.Activation("relu", name=nm("relu1"))(x)

    # Stage 2: replace Conv2D(3x3) with DW3x3 + Ghost 1x1
    x = L.DepthwiseConv2D(3, padding="same", name=nm("dw3b"))(x)
    x = GhostPointwise(num_filters, ratio=ratio, use_bias=False, name=nm("gpw"))(x)

    # Optional: ultra‑light Squeeze-and-Excitation for a small quality boost
    if se_ratio and se_ratio > 0.0:
        se = L.GlobalAveragePooling2D(name=nm("se_gap"))(x)                  # [B, C]
        se = L.Dense(max(1, int(num_filters * se_ratio)), activation="relu",
                     name=nm("se_fc1"))(se)
        se = L.Dense(num_filters, activation="sigmoid", name=nm("se_fc2"))(se)
        se = L.Reshape((1, 1, num_filters), name=nm("se_reshape"))(se)
        x  = L.Multiply(name=nm("se_scale"))([x, se])

    x = L.BatchNormalization(name=nm("bn2"))(x)
    x = L.Activation("relu", name=nm("relu2"))(x)
    return x

    



def pixel_shuffle_up(x, out_channels, name=None):
    nm = (lambda s: None if name is None else f"{name}_{s}")
    # Produce 4*out_channels feature planes (for r=2 pixel shuffle)
    x = L.Conv2D(out_channels * 4, 1, padding="same", groups = 8, name=nm("pre"))(x)
    x = L.Lambda(lambda t: tf.nn.depth_to_space(t, block_size=2), name=nm("ps2"))(x)
    x = L.BatchNormalization(name=nm("bn"))(x)
    x = L.Activation("relu", name=nm("relu"))(x)
    return x

# Decoder Block

def decoder_block(x, s, num_filters, attention=True):
    if attention == True:
        #x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(x)
        
        x = pixel_shuffle_up(x, num_filters)
        
        # UNIQUE name for this stage (e.g., use a running uid)
        ag_name = f"ag_d_true_{K.get_uid('ag_d_true')}"
        s = GhostHyperAttentionGate(
                num_filters=num_filters,
                ratio=2,
                use_hyper=True,
                hyper_gain_init=0.5,
                name=ag_name
            )([x, s])
        x = Concatenate()([x, s])
        x = conv_block(x, num_filters)
        x = GatedDeeperAttentionLayer(units=num_filters, temperature_init=2.0, lmbda_init=5e-5)([x, s])
        return x

    elif attention == 'else':
        #x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(x)

        x = pixel_shuffle_up(x, num_filters)
        
        s = s[:, :, :, :-1]
        s = repeat_elem(s, rep=32)

        ag_name = f"ag_d_else_{K.get_uid('ag_d_else')}"
        ## HyGAM
        s = GhostHyperAttentionGate(
                num_filters=num_filters,
                ratio=2,
                use_hyper=True,
                hyper_gain_init=0.5,
                name=ag_name
            )([x, s])

        x = Concatenate()([x, s])
        ## GCAB
        x = conv_block(x, num_filters)

        ## DuSRA
        x = GatedDeeperAttentionLayer(units=num_filters, temperature_init=2.0, lmbda_init=5e-5)([x, s])
        return x

    else:
        
        #x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(x)

        x = pixel_shuffle_up(x, num_filters)
        
        s = repeat_elem(s, rep=2)

        ag_name = f"ag_d_other_{K.get_uid('ag_d_other')}"
        s = GhostHyperAttentionGate(
                num_filters=num_filters,
                ratio=2,
                use_hyper=True,
                hyper_gain_init=0.5,
                name=ag_name
            )([x, s])

        x = Concatenate()([x, s])
        x = conv_block(x, num_filters)
        x = GatedDeeperAttentionLayer(units=num_filters, temperature_init=2.0, lmbda_init=5e-5)([x, s])
        return x


from tensorflow.keras.applications import EfficientNetB0, EfficientNetB1, MobileNetV2

def build_resnet50_unet(input_shape):
    """ Input """
    inputs = Input(input_shape)

    """ Pre-trained ResNet50 Model """
    resnet50 = MobileNetV2(include_top=False, weights="imagenet", input_tensor=inputs)
    
    """ Encoder """
    s1 = resnet50.get_layer("block_1_expand_relu").output ## (256 x 256x3)
    #s1 = DeeperAttentionLayer(units=3, use_scale=True)([s1, s1])
    print('eff_s1:', s1.shape)
    
    s2 = resnet50.get_layer("block_3_expand_relu").output ## (128 x 128)
    print('eff_s2:', s2.shape)
    #s2 = DeeperAttentionLayer(units=64, use_scale=True)([s2, s2])
    
    
    s3 = resnet50.get_layer("block_6_expand_relu").output ## (64 x 64)
    print('eff_s3:', s3.shape)
    #s3 = DeeperAttentionLayer(units=256, use_scale=True)([s3, s3])
    

    
    s4 = resnet50.get_layer("block_13_expand_relu").output ## (32 x 32)
    print('eff_s4:', s4.shape)
    #s4 = DeeperAttentionLayer(units=512, use_scale=True)([s4, s4])
    

    """ Bridge """
    b1 = resnet50.get_layer("block_16_project").output ## (32 x 32)
    print('eff_b1:', b1.shape)
    #b1 = DeeperAttentionLayer(units=1024, use_scale=True)([b1, b1])
    

    """ Decoder """
    print("d1")
    #b1, s4 = DeeperAttentionLayer(units=512, use_scale=True)([b1, s4])
    
    #s4 = Conv2D(64, 1, activation='relu', padding='same')(s4)

    s4 = GhostPointwise(64, ratio=2, use_bias=False)(s4)
    
    d1 = decoder_block(b1, s4, 64) ## (32 x32)
    #d1 = DeeperAttentionLayer(units=512, use_scale=True)([d1, d1])
    print("d2")
    #d1, s3 = DeeperAttentionLayer(units=256, use_scale=True)([d1, s3])
    
    #s3 = Conv2D(64, 1, activation='relu', padding='same')(s3)

    s3 = GhostPointwise(64, ratio=2, use_bias=False)(s3)
    
    d2 = decoder_block(d1, s3, 64) ## (64 x 64)
    
    #d2 = DeeperAttentionLayer(units=256, use_scale=True)([d2, d2])
    print("d3")
    #d2, s2 = DeeperAttentionLayer(units=128, use_scale=True)([d2, s2])
    
    #s2 = Conv2D(16, 1, activation='relu', padding='same')(s2)

    s2 = GhostPointwise(16, ratio=2, use_bias=False)(s2)
    
    d3 = decoder_block(d2, s2, 32,attention=False) ## (128 x 128)
    
    #d3 = DeeperAttentionLayer(units=128, use_scale=True)([d3, d3])
    print("d4")
    #d3, s1 = DeeperAttentionLayer(units=128, use_scale=True)([d3, s1])
    
    #s1 = Conv2DTranspose(2, (2, 2), strides=2, padding="same")(s1)
    #print(' agian d3 shape:', d3.shape)
    #print(' agian s1 shape:', s1.shape)
    
    #s1 = Conv2D(3, 1, activation='relu', padding='same')(s1)

    s1 = GhostPointwise(3, ratio=2, use_bias=False)(s1)
    
    d4 = decoder_block(d3, s1, 64, attention='else') ## (256 x 256)
    #d4 = DeeperAttentionLayer(units=64, use_scale=True)([d4, d4])
    print('d1 shape:', d1.shape)
    print('d2 shape:', d2.shape)
    print('d3 shape:', d3.shape)
    print('d4 shape:', d4.shape)

    
    '''d1 = Conv2DTranspose(512, (2, 2), strides=2, padding="same")(d1)
    d2 = Conv2DTranspose(256, (2, 2), strides=2, padding="same")(d2)
    d3 = Conv2DTranspose(128, (2, 2), strides=2, padding="same")(d3)
    d4 = Conv2DTranspose(64, (2, 2), strides=2, padding="same")(d4)'''

    d1 = pixel_shuffle_up(d1, 512, name="up_d1_ps")
    d2 = pixel_shuffle_up(d2, 256, name="up_d2_ps")
    d3 = pixel_shuffle_up(d3, 128, name="up_d3_ps")
    d4 = pixel_shuffle_up(d4,  64, name="up_d4_ps")

    

    '''d1 = UpSampling2D(size=(2, 2))(d1)
    d1 = Conv2D(512, (2, 2), padding="same")(d1)
    
    d2 = UpSampling2D(size=(2, 2))(d2)
    d2 = Conv2D(256, (2, 2), padding="same")(d2)
    
    d3 = UpSampling2D(size=(2, 2))(d3)
    d3 = Conv2D(128, (2, 2), padding="same")(d3)
    
    d4 = UpSampling2D(size=(2, 2))(d4)
    d4 = Conv2D(64, (2, 2), padding="same")(d4)'''
    
    print('d1 shape:', d1.shape)
    print('d2 shape:', d2.shape)
    print('d3 shape:', d3.shape)
    print('d4 shape:', d4.shape)
    
    nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4 = fusion_tasks(d1, d2, d3, d4)

    """ Output """
    #outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)

    nestnet_output_1 = Conv2D(1, 1, activation='sigmoid', name='output_5', padding='same')(nestnet_output_1)
    print('output_1:', nestnet_output_1.shape)
    nestnet_output_2 = Conv2D(1, 1, activation='sigmoid', name='output_6', padding='same' )(nestnet_output_2)
    print('output_2:', nestnet_output_2.shape)
    nestnet_output_3 = Conv2D(1, 1, activation='sigmoid', name='output_7', padding='same')(nestnet_output_3)
    print('output_3:', nestnet_output_3.shape)
    nestnet_output_4 = Conv2D(1, 1, activation='sigmoid', name='output_8', padding='same')(nestnet_output_4)
    print('output_4:', nestnet_output_4.shape)
    

    #model = Model(inputs, outputs, name="ResNet50_U-Net")
    model = Model(inputs, [nestnet_output_1, nestnet_output_2, nestnet_output_3, nestnet_output_4], name="ResNet50_U-Net")

    return model

input_shape = (512,512, 3)
model = build_resnet50_unet(input_shape)
model.summary()

In [ ]:
input_shape = (256,256, 3)
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization, Add, Concatenate

def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(
        [tf.TensorSpec([1, *inputs.shape[1:]]) for inputs in model.inputs]
    )
    
    # Calculate FLOPs using TensorFlow's built-in profiler
    graph = concrete_func.graph
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    
    flops = tf.compat.v1.profiler.profile(
        graph,
        run_meta=run_meta,
        cmd='op',
        options=opts
    )
    
    return flops.total_float_ops

# Load your trained model or build it using the provided code
# model = ... (your model loading/building code here)

# Calculate total FLOPs
total_flops = get_flops(model)
gflops = total_flops / 1e9

print(f"Total FLOPs: {total_flops:,}")
print(f"GFLOPs: {gflops:.2f}")

In [ ]:
input_shape = (384,384, 3)
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization, Add, Concatenate

def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(
        [tf.TensorSpec([1, *inputs.shape[1:]]) for inputs in model.inputs]
    )
    
    # Calculate FLOPs using TensorFlow's built-in profiler
    graph = concrete_func.graph
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    
    flops = tf.compat.v1.profiler.profile(
        graph,
        run_meta=run_meta,
        cmd='op',
        options=opts
    )
    
    return flops.total_float_ops

# Load your trained model or build it using the provided code
# model = ... (your model loading/building code here)

# Calculate total FLOPs
total_flops = get_flops(model)
gflops = total_flops / 1e9

print(f"Total FLOPs: {total_flops:,}")
print(f"GFLOPs: {gflops:.2f}")

In [ ]:
input_shape = (512,512, 3)
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization, Add, Concatenate

def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(
        [tf.TensorSpec([1, *inputs.shape[1:]]) for inputs in model.inputs]
    )
    
    # Calculate FLOPs using TensorFlow's built-in profiler
    graph = concrete_func.graph
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    
    flops = tf.compat.v1.profiler.profile(
        graph,
        run_meta=run_meta,
        cmd='op',
        options=opts
    )
    
    return flops.total_float_ops

# Load your trained model or build it using the provided code
# model = ... (your model loading/building code here)

# Calculate total FLOPs
total_flops = get_flops(model)
gflops = total_flops / 1e9

print(f"Total FLOPs: {total_flops:,}")
print(f"GFLOPs: {gflops:.2f}")

In [ ]:
## 224, 224, 3
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization, Add, Concatenate

def get_flops(model):
    concrete = tf.function(lambda inputs: model(inputs))
    concrete_func = concrete.get_concrete_function(
        [tf.TensorSpec([1, *inputs.shape[1:]]) for inputs in model.inputs]
    )
    
    # Calculate FLOPs using TensorFlow's built-in profiler
    graph = concrete_func.graph
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    
    flops = tf.compat.v1.profiler.profile(
        graph,
        run_meta=run_meta,
        cmd='op',
        options=opts
    )
    
    return flops.total_float_ops

# Load your trained model or build it using the provided code
# model = ... (your model loading/building code here)

# Calculate total FLOPs
total_flops = get_flops(model)
gflops = total_flops / 1e9

print(f"Total FLOPs: {total_flops:,}")
print(f"GFLOPs: {gflops:.2f}")

In [ ]:
smooth=1.
#-----------------------------------------------------------------------------------------------------------------------------------------------------------#
'''Function for returning dice coefficient'''
def DICE_COEFF(y_true, y_pred):
    y_true = K.flatten(y_true)
    y_pred = K.flatten(y_pred)
    intersection = K.sum(y_true * y_pred)
    union = K.sum(y_true) + K.sum(y_pred)
    return (2.0 * intersection + smooth) / (union + smooth)
#-----------------------------------------------------------------------------------------------------------------------------------------------------------#

def DICE_COEFF(y_true, y_pred):
    y_true = tf.reshape(y_true, [-1])  # Flatten using tf.reshape
    y_pred = tf.reshape(y_pred, [-1])  # Flatten using tf.reshape
    intersection = tf.reduce_sum(y_true * y_pred)  # Use tf.reduce_sum
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)


'''Dice Coefficient Loss'''
def dice_coef_loss(y_true, y_pred):
    return 1 - DICE_COEFF(y_true, y_pred)

#----------------------------------------------------------------------------------------------------------------------------------------------------------#
'''Function for combining binary cross entropy with dice coeffcients for loss function'''
def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()
    return dice_coef_loss(y_true, y_pred) + bce(y_true, y_pred)

#----------------------------------------------------------------------------------------------------------------------------------------------------------#

'''Function for Jacards coefficient'''
def IOU_JACARD(y_true, y_pred):
    y_true=K.flatten(y_true)
    y_pred=K.flatten(y_pred)
    intersection = K.sum(y_true * y_pred)
    sum_jac = K.sum(y_true + y_pred)
    jac = (intersection + smooth) / (sum_jac - intersection + smooth)
    return jac

def bce_dice_IOU_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()
    return dice_coef_loss(y_true, y_pred) + IOU_JACARD(y_true, y_pred) + bce(y_true, y_pred)

In [ ]:
import tensorflow as tf
import tensorflow.keras.backend as K

def attention_weighted_dice_loss(y_true, y_pred, alpha=0.5, smooth=1.0):
    """
    Attention-Weighted Dice Loss: Assigns higher loss to uncertain/misclassified pixels.
    
    Parameters:
        y_true (tensor): Ground truth labels.
        y_pred (tensor): Predicted labels.
        alpha (float): Scaling factor for exponential weighting.
        smooth (float): Smoothing factor to avoid division by zero.
    
    Returns:
        Tensor: Computed loss value.
    """
    # Ensure y_true and y_pred have the same shape
    y_true = tf.cast(y_true, dtype=tf.float32)
    y_pred = tf.cast(y_pred, dtype=tf.float32)

    # Compute uncertainty-based weighting (higher for misclassified pixels)
    weight_map = tf.abs(y_true - y_pred)  # Difference map
    weight_map = tf.exp(alpha * weight_map)  # Exponential scaling

    # Flatten for Dice computation
    y_true_flat = tf.reshape(y_true, [-1])
    y_pred_flat = tf.reshape(y_pred, [-1])
    weight_map_flat = tf.reshape(weight_map, [-1])

    # Compute weighted Dice loss
    intersection = tf.reduce_sum(weight_map_flat * y_true_flat * y_pred_flat)
    denominator = tf.reduce_sum(weight_map_flat * y_true_flat) + tf.reduce_sum(weight_map_flat * y_pred_flat)

    return 1 - (2.0 * intersection + smooth) / (denominator + smooth + K.epsilon())


#import tensorflow_addons as tfa

import tensorflow as tf

import tensorflow as tf

def boundary_aware_dice_loss(y_true, y_pred):
    """
    Boundary-Aware Dice Loss: Adds edge awareness by computing Sobel gradients.
    """
    smooth = 1.0

    # Ensure input has the correct shape (batch, height, width, channels)
    y_true = tf.ensure_shape(y_true, [None, None, None, 1])
    y_pred = tf.ensure_shape(y_pred, [None, None, None, 1])

    # Compute Sobel edges for boundary awareness
    y_true_edges = tf.image.sobel_edges(y_true)  # Shape: (batch, height, width, 1, 2)
    y_pred_edges = tf.image.sobel_edges(y_pred)

    # Convert shape (batch, height, width, 1, 2) -> (batch, height, width, 2)
    y_true_edges = tf.squeeze(y_true_edges, axis=-2)
    y_pred_edges = tf.squeeze(y_pred_edges, axis=-2)

    # Compute Dice loss based on edges
    intersection = tf.reduce_sum(y_true_edges * y_pred_edges)
    return 1 - (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_edges) + tf.reduce_sum(y_pred_edges) + smooth)


def focal_dice_loss(y_true, y_pred, gamma=2.0):
    """
    Focal Dice Loss: Adds a focal component to Dice Loss to handle class imbalance.
    """
    dice = DICE_COEFF(y_true, y_pred)
    focal_weight = tf.pow(1 - dice, gamma)  # Focal term
    return focal_weight * (1 - dice)

'''def multi_scale_dice_loss(y_true, y_pred):
    """
    Multi-Scale Dice Loss: Computes loss at multiple scales to capture fine & coarse features.
    """
    y_pred_down = tf.image.resize(y_pred, (y_pred.shape[1]//2, y_pred.shape[2]//2))  # Downsample
    y_true_down = tf.image.resize(y_true, (y_true.shape[1]//2, y_true.shape[2]//2))

    loss_high = dice_coef_loss(y_true, y_pred)
    loss_low = dice_coef_loss(y_true_down, y_pred_down)

    return 0.5 * loss_high + 0.5 * loss_low  # Balance loss at different scales
'''
def novel_combined_loss(y_true, y_pred):
    """
    Novel Hybrid Loss: Combines attention-weighted, focal, boundary-aware, and BCE loss.
    """
    bce = tf.keras.losses.BinaryCrossentropy()
    return (0.2 * attention_weighted_dice_loss(y_true, y_pred) +
            0.4 * focal_dice_loss(y_true, y_pred) +
            #0.1 * boundary_aware_dice_loss(y_true, y_pred) + #0.125 * multi_scale_dice_loss(y_true, y_pred) +
            0.2 * bce(y_true, y_pred))  # BCE for stability

In [ ]:
# ===================== RL-WEIGHTED PER-HEAD LOSSES =======================
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K
import tensorflow.keras.layers as L
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import Callback

bce_obj = BinaryCrossentropy(from_logits=False,
                             reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE)

# Logits updated by the controller each epoch
comp_logits = tf.Variable([np.log(0.33), np.log(0.34), np.log(0.33)],
                          dtype=tf.float32, name='rl_comp_logits')  # [attDice, focalDice, BCE]
head_logits = tf.Variable([0., 0., 0., 0.], dtype=tf.float32, name='rl_head_logits')  # 4 heads

def _component_mix(y_true, y_pred):
    w = tf.nn.softmax(comp_logits)  # [3]
    l1 = attention_weighted_dice_loss(y_true, y_pred)
    l2 = focal_dice_loss(y_true, y_pred)
    l3 = bce_obj(y_true, y_pred)
    return w[0]*l1 + w[1]*l2 + w[2]*l3

def make_head_loss(head_index):
    def loss_fn(y_true, y_pred):
        hw = tf.nn.softmax(head_logits)[head_index]
        return hw * _component_mix(y_true, y_pred)
    return loss_fn

losses = [make_head_loss(i) for i in range(4)]  # <- one loss per output head
# ========================================================================


In [ ]:
# ==================== CONTROL-THEORY + BANDIT CONTROLLER ====================
import time

def _project_monotone_simplex(v, increasing=True):
    """
    L2 projection onto {x>=0, sum x = 1, x1 <= x2 <= ...} (if increasing).
    Implements PAVA + renormalization. Used as a deep-supervision prior.
    """
    v = np.array(v, dtype=np.float64)
    if not increasing:
        v = v[::-1]
    n = len(v)
    blocks = [(i, i, v[i]) for i in range(n)]
    k = 0
    while k < len(blocks) - 1:
        if blocks[k][2] <= blocks[k+1][2]:
            k += 1
        else:
            i1,j1,a1 = blocks[k]
            i2,j2,a2 = blocks[k+1]
            avg = (a1*(j1-i1+1) + a2*(j2-i2+1)) / (j2-i1+1)
            blocks[k] = (i1, j2, avg)
            del blocks[k+1]
            k = max(k-1, 0)
    x = np.zeros(n)
    for i1,j1,avg in blocks:
        x[i1:j1+1] = avg
    if not increasing:
        x = x[::-1]
    x = np.maximum(x, 0.0)
    s = x.sum()
    return x/s if s>0 else np.ones_like(x)/len(x)

class ControlBanditPID(Callback):
    """
    Hybrid controller:
      - Outer EXP3-IX bandit samples *discrete* base distributions for component + head weights.
      - Inner PID fine-tunes *continuous* biases to track a target validation Dice setpoint.
      - Separate PI adjusts gate penalties (ConcreteGate.lmbda_var) to meet an epoch-time budget.
    """

    def __init__(self,
                 comp_logits_var, head_logits_var,
                 # --- bandit (EXP3-IX) ---
                 comp_pool=None, head_pool=None, gamma=0.07, eta=0.5, ix_bias=0.05, seed=123,
                 # --- PID target on validation Dice ---
                 target_dice=0.88,
                 # component PID
                 Kp_comp=2.0, Ki_comp=0.4, Kd_comp=0.0, ctrl_lr_comp=0.15, bias_clip_comp=1.5,
                 # head PID
                 Kp_head=1.5, Ki_head=0.3, Kd_head=0.0, ctrl_lr_head=0.15, bias_clip_head=1.5,
                 # desired shift directions when below target
                 comp_dir=np.array([+1.0, +0.4, -1.0]),       # push to Dice > Focal > BCE
                 head_dir=np.array([-1.0, -0.3, +0.3, +1.0]), # shift toward deeper heads
                 # --- compute budget PI (sec/epoch) for gating penalty ---
                 budget_epoch_seconds=None, Kp_gate=0.8, Ki_gate=0.2, gate_lr=1.0e-4,
                 gate_lmbda_bounds=(1e-6, 5e-3),
                 # --- deep supervision prior ---
                 head_monotone='increasing'   # deeper heads >= shallow heads
                 ):
        super().__init__()
        self.comp_logits = comp_logits_var
        self.head_logits = head_logits_var

        # Pools (sum to 1)
        self.comp_pool = comp_pool or [
            np.array([0.60, 0.30, 0.10]),
            np.array([0.40, 0.40, 0.20]),
            np.array([0.25, 0.50, 0.25]),
            np.array([0.20, 0.30, 0.50]),
        ]
        self.head_pool = head_pool or [
            np.array([0.40, 0.30, 0.20, 0.10]),
            np.array([0.25, 0.25, 0.25, 0.25]),
            np.array([0.10, 0.20, 0.30, 0.40]),
        ]

        # Bandit params
        self.gamma = float(gamma)
        self.eta   = float(eta)
        self.ix_bias = float(ix_bias)
        self.rng = np.random.default_rng(seed)
        self.Kc = len(self.comp_pool)
        self.Kh = len(self.head_pool)
        self.w_comp = np.ones(self.Kc, dtype=np.float64)
        self.w_head = np.ones(self.Kh, dtype=np.float64)

        # PID targets + params
        self.target_dice = float(target_dice)

        self.Kp_comp, self.Ki_comp, self.Kd_comp = map(float, (Kp_comp, Ki_comp, Kd_comp))
        self.Kp_head, self.Ki_head, self.Kd_head = map(float, (Kp_head, Ki_head, Kd_head))
        self.ctrl_lr_comp, self.ctrl_lr_head    = float(ctrl_lr_comp), float(ctrl_lr_head)
        self.bias_clip_comp, self.bias_clip_head= float(bias_clip_comp), float(bias_clip_head)

        self.comp_dir = np.array(comp_dir, dtype=np.float64)
        self.comp_dir = self.comp_dir / (np.linalg.norm(self.comp_dir) + 1e-8)
        self.head_dir = np.array(head_dir, dtype=np.float64)
        self.head_dir = self.head_dir / (np.linalg.norm(self.head_dir) + 1e-8)

        # Integrator / derivative states (with simple anti-windup clamps)
        self.bias_comp = np.zeros(3, dtype=np.float64)
        self.bias_head = np.zeros(4, dtype=np.float64)
        self.e_int_comp = 0.0; self.e_prev_comp = 0.0
        self.e_int_head = 0.0; self.e_prev_head = 0.0
        self._int_clip = 50.0  # integral windup guard

        # Compute-time budget PI
        self.budget_epoch_seconds = budget_epoch_seconds
        self.Kp_gate, self.Ki_gate = float(Kp_gate), float(Ki_gate)
        self.gate_lr = float(gate_lr)
        self.gate_min, self.gate_max = gate_lmbda_bounds
        self.e_int_gate = 0.0
        self.t0 = None

        self.head_monotone = head_monotone
        self.idx_c = None; self.idx_h = None
        self.p_c = None;   self.p_h = None
        self.gate_layers = []

    # ---------- helpers ----------
    def _sample_ix(self, w):
        p = (1.0 - self.gamma) * (w / w.sum()) + self.gamma / len(w)
        idx = self.rng.choice(len(w), p=p)
        return idx, p

    def _set_logits_with_bias(self, comp_base, head_base):
        # Components: logits = log(base) + bias_comp
        comp_logits_val = np.log(comp_base + 1e-8) + self.bias_comp
        tf.keras.backend.set_value(self.comp_logits, comp_logits_val.astype(np.float32))

        # Heads: logits = log(project_monotone(softmax(log(base)+bias_head)))
        head_logits_raw = np.log(head_base + 1e-8) + self.bias_head
        head_probs = np.exp(head_logits_raw - np.max(head_logits_raw))
        head_probs = head_probs / np.sum(head_probs)
        head_probs = _project_monotone_simplex(head_probs, increasing=(self.head_monotone=='increasing'))
        tf.keras.backend.set_value(self.head_logits, np.log(head_probs + 1e-8).astype(np.float32))

    def _get_val_dice(self, logs):
        dice_vals = [v for k, v in (logs or {}).items() if k.startswith('val_') and 'dice' in k.lower()]
        if not dice_vals:
            return 1.0 / (1.0 + float((logs or {}).get('val_loss', 0.0)))
        return float(np.mean(dice_vals))

    '''def _collect_gate_layers(self):
        # Find all ConcreteGate instances (nested) once
        self.gate_layers = []
        for m in self.model.submodules:  # includes nested layers
            if isinstance(m, ConcreteGate):
                self.gate_layers.append(m)'''

    def _collect_gate_layers(self):
        """Collect all ConcreteGate instances robustly across Keras versions."""
        gates = []
    
        # Preferred: recursive flatten (available in many TF/Keras versions)
        try:
            all_layers = self.model._flatten_layers(include_self=True, recursive=True)
            for layer in all_layers:
                if isinstance(layer, ConcreteGate):
                    gates.append(layer)
                # If a composite wrapper is present, also grab its inner gate
                if isinstance(layer, GatedDeeperAttentionLayer):
                    gates.append(layer.gate)
        except Exception:
            # Fallback: manual DFS over .layers
            def visit(layer):
                if isinstance(layer, ConcreteGate):
                    gates.append(layer)
                if isinstance(layer, GatedDeeperAttentionLayer):
                    gates.append(layer.gate)
                if hasattr(layer, "layers") and isinstance(layer.layers, (list, tuple)):
                    for child in layer.layers:
                        visit(child)
    
            for root in getattr(self.model, "layers", []):
                visit(root)
    
        # Deduplicate while preserving order
        uniq = {}
        for g in gates:
            uniq[id(g)] = g
        self.gate_layers = list(uniq.values())


    # ---------- Keras hooks ----------
    def on_train_begin(self, logs=None):
        self._collect_gate_layers()

    def on_epoch_begin(self, epoch, logs=None):
        # Bandit sampling
        self.idx_c, self.p_c = self._sample_ix(self.w_comp)
        self.idx_h, self.p_h = self._sample_ix(self.w_head)
        comp_base = self.comp_pool[self.idx_c]
        head_base = self.head_pool[self.idx_h]
        # Apply base + current PID biases
        self._set_logits_with_bias(comp_base, head_base)
        self.t0 = time.time()

    def on_epoch_end(self, epoch, logs=None):
        # ----- reward & PID error -----
        r_dice = self._get_val_dice(logs)
        e = self.target_dice - r_dice  # positive => below target

        # PID (components)
        de_comp = e - self.e_prev_comp
        self.e_int_comp = np.clip(self.e_int_comp + e, -self._int_clip, +self._int_clip)
        u_comp = self.Kp_comp*e + self.Ki_comp*self.e_int_comp + self.Kd_comp*de_comp
        self.bias_comp += self.ctrl_lr_comp * u_comp * self.comp_dir
        self.bias_comp = np.clip(self.bias_comp, -self.bias_clip_comp, +self.bias_clip_comp)
        self.e_prev_comp = e

        # PID (heads)
        de_head = e - self.e_prev_head
        self.e_int_head = np.clip(self.e_int_head + e, -self._int_clip, +self._int_clip)
        u_head = self.Kp_head*e + self.Ki_head*self.e_int_head + self.Kd_head*de_head
        self.bias_head += self.ctrl_lr_head * u_head * self.head_dir
        self.bias_head = np.clip(self.bias_head, -self.bias_clip_head, +self.bias_clip_head)
        self.e_prev_head = e

        # Re-apply logits after PID updates (keep current bandit bases)
        comp_base = self.comp_pool[self.idx_c]
        head_base = self.head_pool[self.idx_h]
        self._set_logits_with_bias(comp_base, head_base)

        # ----- compute-time PI control on gate penalties -----
        if self.budget_epoch_seconds is not None and self.t0 is not None and len(self.gate_layers) > 0:
            dur = time.time() - self.t0
            err_t = dur - self.budget_epoch_seconds  # positive => too slow (increase penalty)
            self.e_int_gate = np.clip(self.e_int_gate + err_t, -10*self.budget_epoch_seconds, +10*self.budget_epoch_seconds)
            delta = self.Kp_gate*err_t + self.Ki_gate*self.e_int_gate
            for g in self.gate_layers:
                new_lmbda = float(g.lmbda_var.numpy()) + self.gate_lr * delta
                new_lmbda = float(np.clip(new_lmbda, self.gate_min, self.gate_max))
                g.lmbda_var.assign(new_lmbda)

        # ----- EXP3-IX weight update on arm distributions -----
        r = float(np.clip(r_dice, 0.0, 1.0))
        est_c = r / (self.p_c[self.idx_c] + 1e-6 + self.ix_bias)
        est_h = r / (self.p_h[self.idx_h] + 1e-6 + self.ix_bias)
        self.w_comp[self.idx_c] *= np.exp(self.eta * est_c / self.Kc)
        self.w_head[self.idx_h] *= np.exp(self.eta * est_h / self.Kh)

        # Optional logs
        logs = logs or {}
        logs['ctrl_dice'] = r_dice
        logs['ctrl_err']  = e
        if self.budget_epoch_seconds is not None and self.t0 is not None:
            logs['epoch_sec'] = time.time() - self.t0
# ================================================================================


In [ ]:
EPOCHS =200
BATCH_SIZE = 32
learning_rate = 0.5e-3
IMAGE_SIZE = (224, 224)
train_generator_args = dict(rotation_range=0.1,
                            width_shift_range=0.05,
                            height_shift_range=0.05,
                            shear_range=0.05,
                            zoom_range=0.05,
                            horizontal_flip=True,
                            vertical_flip=True,
                            fill_mode='nearest')

train_gen = train_generator(df_train, BATCH_SIZE, train_generator_args, target_size=IMAGE_SIZE)
val_gen = train_generator(df_val, BATCH_SIZE, dict(), target_size=IMAGE_SIZE)

# 修改损失函数中的 BinaryCrossentropy from_logits=False
def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)  # Ensure from_logits=False
    return dice_coef_loss(y_true, y_pred) + bce(y_true, y_pred)

# 优化器
opt = Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.999, epsilon=1e-7, amsgrad=False)

# 模型编译
#model.compile(optimizer=opt, loss=novel_combined_loss, metrics=[[iou, dice_coef], [iou, dice_coef], [iou, dice_coef], [iou, dice_coef]])
model.compile(
    optimizer=opt,
    loss=losses,  # <- per-head losses that the controller adjusts via logits
    metrics=[[iou, dice_coef], [iou, dice_coef], [iou, dice_coef], [iou, dice_coef]]
)


In [ ]:
callbacks = [
    ControlBanditPID(
        comp_logits_var=comp_logits,
        head_logits_var=head_logits,
        # bandit exploration
        gamma=0.07, eta=0.5, ix_bias=0.05,
        # control-theory setpoints
        target_dice=0.88,           # <-- pick a reachable setpoint (e.g., 0.85–0.90)
        budget_epoch_seconds=None,  # e.g., 55.0 to enforce an epoch time budget; None disables
    ),
    ModelCheckpoint('RESUNET1_skin_seg.keras', verbose=1, save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=50, verbose=1, min_lr=1e-6),
    EarlyStopping(monitor='val_loss', restore_best_weights=True, patience=60, verbose=1),
]


In [ ]:
import tensorflow as tf
IMAGE_H, IMAGE_W = IMAGE_SIZE  # e.g., (224, 224)

def make_wrapped_dataset(base_gen):
    def _gen():
        for x, y in base_gen:
            yield tf.cast(x, tf.float32), (
                tf.cast(y, tf.float32),
                tf.cast(y, tf.float32),
                tf.cast(y, tf.float32),
                tf.cast(y, tf.float32),
            )
    output_signature = (
        tf.TensorSpec(shape=(None, IMAGE_H, IMAGE_W, 3), dtype=tf.float32),
        (
            tf.TensorSpec(shape=(None, IMAGE_H, IMAGE_W, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(None, IMAGE_H, IMAGE_W, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(None, IMAGE_H, IMAGE_W, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(None, IMAGE_H, IMAGE_W, 1), dtype=tf.float32),
        )
    )
    return tf.data.Dataset.from_generator(_gen, output_signature=output_signature).prefetch(tf.data.AUTOTUNE)

train_ds = make_wrapped_dataset(train_gen)
val_ds   = make_wrapped_dataset(val_gen)

history = model.fit(
    train_ds,
    steps_per_epoch=int(len(df_train) / BATCH_SIZE),
    epochs=EPOCHS,
    callbacks=callbacks,
    validation_data=val_ds,
    validation_steps=int(len(df_val) / BATCH_SIZE)
)
